In [51]:
import os
import pandas as pd
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from sklearn.metrics import silhouette_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer

In [27]:
# Pandas 출력 옵션 조정 - All
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [29]:
# Pandas 출력 옵션 조정 - Default
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

In [2]:
## Defines
# Data confirmmation :: Nan counts + Nan Ratios, Unique values
def analyse_dataframe(df):
    # NaN 개수, 비율 및 고유 값 개수 계산
    nan_counts = df.isnull().sum()
    nan_ratios = df.isnull().mean() * 100  # 백분율로 표시
    unique_values_count = df.nunique()

    # 결과 데이터 프레임 생성
    analysis_result = pd.DataFrame({
        'NaN Counts': nan_counts,
        'NaN Ratios (%)': nan_ratios,
        'Unique Values Count': unique_values_count
    })

    return analysis_result

# 지정된 칼럼의 NaN 값을 해당 칼럼의 평균값으로 대입
def fill_na_with_mean(df, columns, decimal_places=None):
    for column in columns:
        mean_value = df[column].mean()
        if decimal_places is not None:
            mean_value = round(mean_value, decimal_places)
        elif decimal_places is None and column in integer_columns:
            mean_value = int(round(mean_value))
        df[column] = df[column].fillna(mean_value)
    return df

In [3]:
# Configs
base = r'C:\Dev_AIWM\jupyter\KNN'
cbk = f'{base}\\1212_M02'
raw_path1 = f'{base}\\1212_M02\\dataset'
raw_path = f'{base}\\olds\\raw_data'
crt_path = f'{base}\\1212_M02\\creatures'

In [4]:
req_cols = [
'sex_sys_val', 'birthdt'    # 'birthdt_sys_val' 없음, 'birthdt'
, 'MAGE', 'GRAN', 'PARN', 'AMNI', 'METC', 'FETC', 'FETC2'
, 'MULG', 'BIR1', 'BIR2', 'BIR3', 'BIR4', 'PREP', 'DM', 'HTN', 'CHOR', 'PROM'
, 'STER', 'ATBYN'
, 'BIRD', 'BPIA', 'ADMD', 'GAGEW', 'GAGED'
, 'APGS1', 'APGUK1', 'APGS5', 'APGUK5', 'RESU'
, 'BWEI', 'BHEI', 'BHEIUK', 'BHEAD', 'BHEADUK', 'BTEM', 'BTEMUK', 'BBPH', 'BBPHUK'
, 'SIGN', 'BBBE', 'BBBEUK'
, 'BHTCUT', 'BHDCUT', 'ALS', 'MPH', 'PH'
, 'PHUDSTDT', 'SFT', 'SFTTHDT', 'SFTFUDT', 'SFTUH'
, 'SFTUM', 'SFTUN', 'STRDU'
, 'BIRDH', 'BIRDM'    # EFYDT 추가 (0319)
, 'STRDUOT', 'OXYT28', 'ARRE28', 'DATE36', 'DEATH36', 'OXYT36', 'ARRE36', 'BDP', 'IARVPPD'
, 'NIARVRPD', 'NIARVHFNC', 'AOXYUPPD'
, 'DATE362', 'DELM'    # , 'DTHDT1' 추가 (01/27)
, 'ACL', 'PDAD', 'PDATR', 'PDADDT', 'PDADDTH'
, 'LBP', 'NESE', 'INHG'
, 'PVL', 'IBIF', 'IBIFCMT', 'SEPS', 'BSFDT1'
, 'MENI', 'MENIDT'
, 'INVFPOD', 'NTET', 'NTETDT', 'NTETY', 'NTETDTY', 'NTETOY', 'IPERR', 'IPERRDT', 'IPERROY'
## ------------------ 02/27 Added lists ------------------------------>>
, 'PMIRNSG', 'PMIRNSGND', 'EFYN', 'EFYDT', 'DCD', 'FDCDT', 'BSF1', 'STDAY', 'EYTHTRAN'
## ------------------ 04/15 Added list ------------------------------>>
, 'DEATH'
## ------------------ 02/27 Delete lists ------------------------------>>
# , 'PHUD1', 'PHUD2', 'PHUD3', 'PHUD4', 'PHUD5', 'PHUD6', 'PHUDOT'
# , 'PADDR1', 'PADDR2', 'PDADTY', 'ACLDT'
# , 'LBPD1', 'LBPD2', 'LBPD3', 'LBPDOT'
# , 'BSFFS1', 'BSF2', 'BSFFS2', 'BSF3', 'BSFDT3', 'BSFFS3'
# , 'FSF1', 'FSFDT1', 'FSFFS1', 'FSF2', 'FSFDT2', 'FSFFS2'
# , 'STERP', 'RESUO', 'RESUP', 'RESUI', 'RESUH', 'RESUE', 'RESUC'
# , 'SFTUP', 'SFTUR', 'SFTTH'
# , 'STRDUT1', 'STRDUT2', 'STRDUD1', 'STRDUD2', 'STRDUD3', 'STRDUD4'
# , 'STERD1', 'STERD2', 'STERD4'
# , 'PROMN', 'PROMD', 'PROMH', 'PRHO', 'PRDH'
# , 'MCOU', 'FCOU'
]

# 모든 칼럼 이름을 소문자로 변환
cols = [col.lower() for col in req_cols]
knn_df = pd.read_csv(os.path.join(raw_path1, "KNN_origin.csv"), usecols = cols)
# knn_df = pd.read_excel(os.path.join(raw_path1, "KNN_origin.xlsx"), usecols = cols)
# knn_df = pd.read_excel(os.path.join(raw_path, "(KNN)DATA.xlsx"), usecols = cols)
knn_df

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\3760666957.py:42: DtypeWarning: Columns (59,60,98,113) have mixed types. Specify dtype option on import or set low_memory=False.
  knn_df = pd.read_csv(os.path.join(raw_path1, "KNN_origin.csv"), usecols = cols)


,sex_sys_val,birthdt,mulg,bir1,bir2,bir3,bir4,prep,dm,htn,...,iperrdt,pmirnsg,pmirnsgnd,efyn,efydt,eythtran,fdcdt,stday,dcd,death
0,1,2013-12-18,1,NaN,NaN,NaN,NaN,1,1,2,...,NaN,0.0,NaN,2,2013-12-28,1,2014-02-06,50,1,NaN
1,1,2013-11-15,1,NaN,NaN,NaN,NaN,1,1,1,...,NaN,2.0,NaN,2,2013-12-15,2,2014-02-03,80,1,NaN
2,1,2013-11-15,2,1.0,NaN,NaN,NaN,1,1,1,...,NaN,0.0,NaN,2,2013-11-29,2,2014-01-08,54,1,NaN
3,1,2013-11-15,2,NaN,2.0,NaN,NaN,1,1,1,...,NaN,0.0,NaN,1,NaN,2,2014-01-22,68,2,NaN
4,1,2013-11-11,1,NaN,NaN,NaN,NaN,1,1,1,...,NaN,1.0,NaN,2,2013-12-12,2,2014-02-13,94,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20408,1,2022-11-28,1,NaN,NaN,NaN,NaN,1,1,1,...,NaN,0.0,NaN,2,2022-12-25,1,2023-02-09,73,2,NaN
20409,1,2022-12-27,1,NaN,NaN,NaN,NaN,1,1,1,...,NaN,2.0,NaN,2,2023-01-30,2,2023-03-29,92,1,NaN
20410,2,2022-12-01,1,NaN,NaN,NaN,NaN,2,1,1,...,NaN,0.0,NaN,2,2022-12-06,2,2023-01-14,44,1,NaN
20411,1,2022-11-30,2,NaN,2.0,NaN,NaN,2,2,1,...,NaN,0.0,NaN,2,2022-12-09,1,2023-01-02,33,1,NaN


In [5]:
analyse_dataframe(knn_df)

,NaN Counts,NaN Ratios (%),Unique Values Count
sex_sys_val,0,0.000000,3
birthdt,0,0.000000,3593
mulg,0,0.000000,4
bir1,17067,83.608485,1
bir2,16421,80.443835,1
...,...,...,...
eythtran,0,0.000000,2
fdcdt,0,0.000000,3633
stday,0,0.000000,313
dcd,0,0.000000,4


In [6]:
for column in knn_df.columns:
    print(f"칼럼 이름: {column}")
    print(f"데이터 타입: {knn_df[column].dtype}")
    print(f"유니크한 값:")
    print(knn_df[column].unique())
    print("-" * 40)

칼럼 이름: sex_sys_val
데이터 타입: int64
유니크한 값:
[1 2 3]
----------------------------------------
칼럼 이름: birthdt
데이터 타입: object
유니크한 값:
['2013-12-18' '2013-11-15' '2013-11-11' ... '2022-05-05' '2022-07-09'
 '2022-06-19']
----------------------------------------
칼럼 이름: mulg
데이터 타입: int64
유니크한 값:
[1 2 3 4]
----------------------------------------
칼럼 이름: bir1
데이터 타입: float64
유니크한 값:
[nan  1.]
----------------------------------------
칼럼 이름: bir2
데이터 타입: float64
유니크한 값:
[nan  2.]
----------------------------------------
칼럼 이름: bir3
데이터 타입: float64
유니크한 값:
[nan  3.]
----------------------------------------
칼럼 이름: bir4
데이터 타입: float64
유니크한 값:
[nan  4.]
----------------------------------------
칼럼 이름: prep
데이터 타입: int64
유니크한 값:
[1 2]
----------------------------------------
칼럼 이름: dm
데이터 타입: int64
유니크한 값:
[1 2 3]
----------------------------------------
칼럼 이름: htn
데이터 타입: int64
유니크한 값:
[2 1 3]
----------------------------------------
칼럼 이름: chor
데이터 타입: int64
유니크한 값:
[3 2 1]
---------------------------

In [7]:
## nan / 0 Findings

In [8]:
dfc = knn_df

summary = []

for col in dfc.columns:
    dtype = dfc[col].dtype
    total = len(dfc)
    nan_count = dfc[col].isna().sum()
    nan_ratio = nan_count / total * 100
    zero_count = (dfc[col] == 0).sum() if np.issubdtype(dtype, np.number) else np.nan
    zero_ratio = (zero_count / total * 100) if np.issubdtype(dtype, np.number) else np.nan

    summary.append({
        'column': col,
        'dtype': dtype,
        'nan_count': nan_count,
        'nan_ratio(%)': round(nan_ratio, 2),
        'zero_count': zero_count,
        'zero_ratio(%)': round(zero_ratio, 2) if not pd.isna(zero_ratio) else np.nan
    })

df_summary = pd.DataFrame(summary)
df_summary.to_csv(os.path.join(crt_path, "cols_summary_4Rev.csv"), index=True)
df_summary

,column,dtype,nan_count,nan_ratio(%),zero_count,zero_ratio(%)
0,sex_sys_val,int64,0,0.00,0.0,0.00
1,birthdt,object,0,0.00,NaN,NaN
2,mulg,int64,0,0.00,0.0,0.00
3,bir1,float64,17067,83.61,0.0,0.00
4,bir2,float64,16421,80.44,0.0,0.00
...,...,...,...,...,...,...
101,eythtran,int64,0,0.00,0.0,0.00
102,fdcdt,object,0,0.00,NaN,NaN
103,stday,int64,0,0.00,254.0,1.24
104,dcd,int64,0,0.00,0.0,0.00


In [9]:
knn_df['niarvhfnc']

0        NaN
1        NaN
2        NaN
3        NaN
4        NaN
        ... 
20408    0.0
20409    5.0
20410    9.0
20411    0.0
20412    0.0
Name: niarvhfnc, Length: 20413, dtype: float64

In [10]:
## Imputations

In [11]:
# # 1. 평균값으로 대체할 칼럼들
# mean_cols = ['apgs1', 'apgs5', 'btem', 'bbph', 'bhead', 'bhei']
# knn_df[mean_cols] = knn_df[mean_cols].fillna(knn_df[mean_cols].mean())

# 2. 0으로 대체할 칼럼들
zero_cols = ['ntetoy', 'arre36', 'death36', 'bir1', 'bir2'
             , 'bir3', 'bir4', 'niarvhfnc'
             , 'arre28', 'btemuk'
             , 'bbphuk', 'bbbeuk', 'bheaduk', 'bheiuk'
             , 'sftun', 'pdad'
## ------------------ 02/27 Delete lists ------------------------------>>
#             'strdut1', 'strdut2', 'pdadty', 'paddr1', 'paddr2'
#              , 'sterp', 'strdud1', 'strdud2', 'strdud3', 'strdud4'
#              , 'bsf1', 'bsffs1', 'bsf2', 'bsffs2', 'bsf3', 'bsffs3'
#              , 'lbpd1', 'lbpd2', 'lbpd3', 'fsf1', 'fsffs1', 'fsf2', 'fsffs2'
#              , 'resuc', 'resue', 'resui', 'resuo', 'resuh', 'resup'
#              , 'phud1', 'phud2', 'phud3', 'phud4', 'phud5', 'phud6'
#              , 'sftup', 'sftur', 'sftth'
#              , 'sterd2', 'sterd1', 'sterd4', 'menifs'
#              , 'promh', 'promn', 'prdh'
            ]
knn_df[zero_cols] = knn_df[zero_cols].fillna(0)

# 3. 시간 및 날짜 포맷 관련 처리할 칼럼들
date_cols = ['birthdt', 'ntetdt', 'ntetdty', 'date362', 'date36', 'menidt', 'pdaddt', 'iperrdt'
             , 'bsfdt1', 'fdcdt', 'efydt', 'phudstdt'
## ------------------ 02/27 Delete lists ------------------------------>>
#              , 'acldt', 'bsfdt3', 'fsfdt1', 'fsfdt2', 'promd
            ]
for col in date_cols:
    knn_df[col] = pd.to_datetime(knn_df[col], errors='coerce').dt.date

knn_df['pdaddth'] = knn_df['pdaddth'].apply(
    lambda x: f"{int(x):02}:00" if not pd.isna(x) else None
)


# # 4. prho 처리: 시간 형식으로 변환 및 결측값 NaT 처리
# knn_df['prho'] = knn_df['prho'].apply(
#     lambda x: f"{int(x):02}:00" if not pd.isna(x) else None
# )

# 5. 그 외 값 대입
knn_df['ntety'].fillna(1, inplace=True)
knn_df['oxyt36'].fillna(1, inplace=True)
knn_df['iperroy'].fillna(1, inplace=True)
knn_df['atbyn'].fillna(3, inplace=True)

# 6. sign과 bbbe 처리: {sign}+{bbbe} 생성 후 sign 삭제
knn_df['sign'] = knn_df['sign'].replace({1: '+', 2: '-'})
knn_df['bbbe'] = knn_df['sign'].astype(str) + knn_df['bbbe'].astype(str)
knn_df['bbbe'] = pd.to_numeric(knn_df['bbbe'], errors='coerce')
knn_df['bbbe'].fillna(knn_df['bbbe'].mean(), inplace=True)

# birdm, birdh -> birtm 생성
before_birdm_birdh = len(knn_df)
knn_df.dropna(subset=['birdh', 'birdm'], inplace=True)
after_birdm_birdh = len(knn_df)
print(f"Removed {before_birdm_birdh - after_birdm_birdh} individuals due to missing 'birdh' or 'birdm'.")
knn_df['birdh'] = knn_df['birdh'].astype(int)
knn_df['birdm'] = knn_df['birdm'].astype(int)
knn_df['birtm'] = pd.to_datetime(knn_df['birdh'].astype(str) + ':' + knn_df['birdm'].astype(str), format='%H:%M', errors='coerce')
knn_df['birtm'] = knn_df['birtm'].dt.strftime('%H:%M')

# 'lbpdot', 'phudot'
# 7. 삭제할 칼럼들
drop_cols = ['apguk1', 'apguk5', 'fetc2', 'fetc', 'strduot', 'metc', 'ibifcmt'
             , 'bhdcut', 'bhtcut', 'phudstdt', 'sftum', 'sftuh', 'sftfudt'
             , 'sftthdt', 'birdm', 'birdh', 'sign'
             # , 'lbpdot', 'phudot'
            ]
# knn_df.drop(columns=drop_cols, inplace=True)

# 보조적 산소/유량 투여기간 : 결즉인원 제거
if 'aoxyuppd' in knn_df.columns:
    before_aoxyuppd = len(knn_df)
    knn_df.dropna(subset=['aoxyuppd'], inplace=True)
    after_aoxyuppd = len(knn_df)
    print(f"Removed {before_aoxyuppd - after_aoxyuppd} individuals due to missing 'aoxyuppd'.")

# sex_sys_val
count_sex_val_3 = knn_df['sex_sys_val'].value_counts().get(3, 0) if 'sex_sys_val' in knn_df.columns else "Column 'sex_sys_val' does not exist"
print("n of sex_sys_val Unknown :", count_sex_val_3)
knn_df = knn_df[knn_df['sex_sys_val'] != 3] if 'sex_sys_val' in knn_df.columns else knn_df
print(knn_df[['sex_sys_val']])

# 결과 확인
knn_df.info()

Removed 52 individuals due to missing 'birdh' or 'birdm'.
Removed 1 individuals due to missing 'aoxyuppd'.
n of sex_sys_val Unknown : 8
       sex_sys_val
0                1
1                1
2                1
3                1
4                1
...            ...
20408            1
20409            1
20410            2
20411            1
20412            1

[20352 rows x 1 columns]
<class 'pandas.core.frame.DataFrame'>
Index: 20352 entries, 0 to 20412
Columns: 107 entries, sex_sys_val to birtm
dtypes: float64(38), int32(2), int64(42), object(25)
memory usage: 16.6+ MB


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\344628415.py:44: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  knn_df['ntety'].fillna(1, inplace=True)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\344628415.py:45: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a

In [11]:
'''# 2. 0으로 대체할 칼럼들
zero_cols = ['ntetoy', 'arre36', 'death36', 'bir1', 'bir2'
             , 'bir3', 'bir4', 'niarvhfnc'
             , 'arre28', 'btemuk'
             , 'bbphuk', 'bbbeuk', 'bheaduk', 'bheiuk'
             , 'sftun', 'pdad'
## ------------------ 02/27 Delete lists ------------------------------>>
#             'strdut1', 'strdut2', 'pdadty', 'paddr1', 'paddr2'
#              , 'sterp', 'strdud1', 'strdud2', 'strdud3', 'strdud4'
#              , 'bsf1', 'bsffs1', 'bsf2', 'bsffs2', 'bsf3', 'bsffs3'
#              , 'lbpd1', 'lbpd2', 'lbpd3', 'fsf1', 'fsffs1', 'fsf2', 'fsffs2'
#              , 'resuc', 'resue', 'resui', 'resuo', 'resuh', 'resup'
#              , 'phud1', 'phud2', 'phud3', 'phud4', 'phud5', 'phud6'
#              , 'sftup', 'sftur', 'sftth'
#              , 'sterd2', 'sterd1', 'sterd4', 'menifs'
#              , 'promh', 'promn', 'prdh'
            ]
knn_df[zero_cols] = knn_df[zero_cols].fillna(0)

# 3. 시간 및 날짜 포맷 관련 처리할 칼럼들
date_cols = ['birthdt', 'ntetdt', 'ntetdty', 'date362', 'date36', 'menidt', 'pdaddt', 'iperrdt'
             , 'bsfdt1', 'fdcdt', 'efydt', 'phudstdt'
## ------------------ 02/27 Delete lists ------------------------------>>
#              , 'acldt', 'bsfdt3', 'fsfdt1', 'fsfdt2', 'promd
            ]
for col in date_cols:
    knn_df[col] = pd.to_datetime(knn_df[col], errors='coerce').dt.date

knn_df['pdaddth'] = knn_df['pdaddth'].apply(
    lambda x: f"{int(x):02}:00" if not pd.isna(x) else None
)


# # 4. prho 처리: 시간 형식으로 변환 및 결측값 NaT 처리
# knn_df['prho'] = knn_df['prho'].apply(
#     lambda x: f"{int(x):02}:00" if not pd.isna(x) else None
# )

# 5. 그 외 값 대입
knn_df['ntety'].fillna(1, inplace=True)
knn_df['oxyt36'].fillna(1, inplace=True)
knn_df['iperroy'].fillna(1, inplace=True)
knn_df['atbyn'].fillna(3, inplace=True)

# 6. sign과 bbbe 처리: {sign}+{bbbe} 생성 후 sign 삭제
knn_df['sign'] = knn_df['sign'].replace({1: '+', 2: '-'})
knn_df['bbbe'] = knn_df['sign'].astype(str) + knn_df['bbbe'].astype(str)
knn_df['bbbe'] = pd.to_numeric(knn_df['bbbe'], errors='coerce')
knn_df['bbbe'].fillna(knn_df['bbbe'].mean(), inplace=True)

# birdm, birdh -> birtm 생성
before_birdm_birdh = len(knn_df)
knn_df.dropna(subset=['birdh', 'birdm'], inplace=True)
after_birdm_birdh = len(knn_df)
print(f"Removed {before_birdm_birdh - after_birdm_birdh} individuals due to missing 'birdh' or 'birdm'.")
knn_df['birdh'] = knn_df['birdh'].astype(int)
knn_df['birdm'] = knn_df['birdm'].astype(int)
knn_df['birtm'] = pd.to_datetime(knn_df['birdh'].astype(str) + ':' + knn_df['birdm'].astype(str), format='%H:%M', errors='coerce')
knn_df['birtm'] = knn_df['birtm'].dt.strftime('%H:%M')

# 'lbpdot', 'phudot'
# 7. 삭제할 칼럼들
drop_cols = ['apguk1', 'apguk5', 'fetc2', 'fetc', 'strduot', 'metc', 'ibifcmt'
             , 'bhdcut', 'bhtcut', 'phudstdt', 'sftum', 'sftuh', 'sftfudt'
             , 'sftthdt', 'birdm', 'birdh', 'sign'
             # , 'lbpdot', 'phudot'
            ]
# knn_df.drop(columns=drop_cols, inplace=True)

# 보조적 산소/유량 투여기간 : 결즉인원 제거
if 'aoxyuppd' in knn_df.columns:
    before_aoxyuppd = len(knn_df)
    knn_df.dropna(subset=['aoxyuppd'], inplace=True)
    after_aoxyuppd = len(knn_df)
    print(f"Removed {before_aoxyuppd - after_aoxyuppd} individuals due to missing 'aoxyuppd'.")

# sex_sys_val
count_sex_val_3 = knn_df['sex_sys_val'].value_counts().get(3, 0) if 'sex_sys_val' in knn_df.columns else "Column 'sex_sys_val' does not exist"
print("n of sex_sys_val Unknown :", count_sex_val_3)
knn_df = knn_df[knn_df['sex_sys_val'] != 3] if 'sex_sys_val' in knn_df.columns else knn_df
print(knn_df[['sex_sys_val']])

# 결과 확인
knn_df.info()'''

Removed 52 individuals due to missing 'birdh' or 'birdm'.
Removed 1 individuals due to missing 'aoxyuppd'.
n of sex_sys_val Unknown : 8
       sex_sys_val
0                1
1                1
2                1
3                1
4                1
...            ...
20408            1
20409            1
20410            2
20411            1
20412            1

[20352 rows x 1 columns]
<class 'pandas.core.frame.DataFrame'>
Index: 20352 entries, 0 to 20412
Columns: 107 entries, sex_sys_val to birtm
dtypes: float64(38), int32(2), int64(42), object(25)
memory usage: 16.6+ MB


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_30640\2061347014.py:44: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  knn_df['ntety'].fillna(1, inplace=True)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_30640\2061347014.py:45: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as

'# 2. 0으로 대체할 칼럼들\nzero_cols = [\'ntetoy\', \'arre36\', \'death36\', \'bir1\', \'bir2\'\n             , \'bir3\', \'bir4\', \'niarvhfnc\'\n             , \'arre28\', \'btemuk\'\n             , \'bbphuk\', \'bbbeuk\', \'bheaduk\', \'bheiuk\'\n             , \'sftun\', \'pdad\'\n## ------------------ 02/27 Delete lists ------------------------------>>\n#             \'strdut1\', \'strdut2\', \'pdadty\', \'paddr1\', \'paddr2\'\n#              , \'sterp\', \'strdud1\', \'strdud2\', \'strdud3\', \'strdud4\'\n#              , \'bsf1\', \'bsffs1\', \'bsf2\', \'bsffs2\', \'bsf3\', \'bsffs3\'\n#              , \'lbpd1\', \'lbpd2\', \'lbpd3\', \'fsf1\', \'fsffs1\', \'fsf2\', \'fsffs2\'\n#              , \'resuc\', \'resue\', \'resui\', \'resuo\', \'resuh\', \'resup\'\n#              , \'phud1\', \'phud2\', \'phud3\', \'phud4\', \'phud5\', \'phud6\'\n#              , \'sftup\', \'sftur\', \'sftth\'\n#              , \'sterd2\', \'sterd1\', \'sterd4\', \'menifs\'\n#              , \'promh\', \'pro

In [12]:
# 칼럼 'acl' 처리 --> 3개 칼럼에 대해 확인해서 진행 : 총 5개 칼럼 제작
knn_df['pda_non-treatment'] = ((knn_df['pdad'].isin([1, 0])) &
                                (knn_df['acl'].isin([1, np.nan])) &
                                (knn_df['pdatr'].isin([14, 15]))).astype(int)
knn_df['pda_med-only'] = ((knn_df['pdad'] == 2) &
                          (knn_df['acl'] == 1)).astype(int)
knn_df['pda_op-only'] = ((knn_df['pdad'] == 1) &
                         (knn_df['acl'] == 2)).astype(int)
knn_df['pda_med+op'] = ((knn_df['pdad'] == 2) &
                        (knn_df['acl'] == 2)).astype(int)
knn_df['pda_unknown'] = ((knn_df['pdatr'].isin([12, 13, 16])) &
                         (knn_df['acl'].isin([3, np.nan]))).astype(int)

# 각 칼럼별 1의 개수 출력
counts = knn_df[['pda_non-treatment', 'pda_med-only', 'pda_op-only', 'pda_med+op', 'pda_unknown']
               ].sum()
print(counts)
knn_df[['pda_non-treatment', 'pda_med-only', 'pda_op-only', 'pda_med+op', 'pda_unknown']]

knn_df['PDA_TRD'] = ((knn_df['pda_med-only'] == 1) | 
                      (knn_df['pda_op-only'] == 1) | 
                      (knn_df['pda_med+op'] == 1)
                    ).astype(int)

# 'PDA_TRD'와 'pda_unknown' 출력
print(knn_df[['PDA_TRD', 'pda_unknown']])
counts = knn_df[['PDA_TRD', 'pda_unknown']].sum()
print(counts)

pda_non-treatment    13052
pda_med-only          4758
pda_op-only            567
pda_med+op            1385
pda_unknown            584
dtype: int64
       PDA_TRD  pda_unknown
0            1            0
1            0            0
2            0            0
3            1            0
4            1            0
...        ...          ...
20408        1            0
20409        1            0
20410        0            0
20411        0            0
20412        0            0

[20352 rows x 2 columns]
PDA_TRD        6710
pda_unknown     584
dtype: int64


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\2013537768.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  knn_df['pda_non-treatment'] = ((knn_df['pdad'].isin([1, 0])) &
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\2013537768.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  knn_df['pda_med-only'] = ((knn_df['pdad'] == 2) &
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\2013537768.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sl

In [15]:
# # .dt accessor를 다시 쓰기 전에 datetime 형식으로 재변환
# date_cols_for_dt = [
#     'birthdt', 'ntetdt', 'ntetdty', 'date362', 'date36', 'menidt',
#     'pdaddt', 'iperrdt', 'bsfdt1', 'fdcdt', 'efydt', 'phudstdt'
# ]
# for col in date_cols_for_dt:
#     if col in knn_df.columns:
#         knn_df[col] = pd.to_datetime(knn_df[col], errors='coerce')

# .dt accessor를 다시 쓰기 전에 datetime 형식으로 재변환 + 비교 출력
date_cols_for_dt = [
    'birthdt', 'ntetdt', 'ntetdty', 'date362', 'date36', 'menidt',
    'pdaddt', 'iperrdt', 'bsfdt1', 'fdcdt', 'efydt', 'phudstdt'
]

comparison_results = {}

for col in date_cols_for_dt:
    if col not in knn_df.columns:
        continue

    before = knn_df[col].copy()
    after = pd.to_datetime(knn_df[col], errors='coerce')

    # 실제 적용
    knn_df[col] = after

    cmp_df = pd.DataFrame({
        'before': before,
        'after': after
    })

    before_as_dt = pd.to_datetime(before, errors='coerce')
    same_or_both_missing = before_as_dt.eq(after) | (before_as_dt.isna() & after.isna())
    n_diff = (~same_or_both_missing).sum()

    comparison_results[col] = {
        'before_dtype': before.dtype,
        'after_dtype': knn_df[col].dtype,
        'n_diff': int(n_diff),
        'diff_rows': cmp_df.loc[~same_or_both_missing, ['before', 'after']]
    }

    print(f"\n[{col}]")
    print(f"before dtype: {before.dtype}")
    print(f"after  dtype: {knn_df[col].dtype}")
    print(f"n different rows: {n_diff}")

    if n_diff > 0:
        print(cmp_df.loc[~same_or_both_missing, ['before', 'after']].head(20))
    else:
        print("All values are identical (or both missing).")


[birthdt]
before dtype: object
after  dtype: datetime64[ns]
n different rows: 0
All values are identical (or both missing).

[ntetdt]
before dtype: object
after  dtype: datetime64[ns]
n different rows: 0
All values are identical (or both missing).

[ntetdty]
before dtype: object
after  dtype: datetime64[ns]
n different rows: 0
All values are identical (or both missing).

[date362]
before dtype: object
after  dtype: datetime64[ns]
n different rows: 0
All values are identical (or both missing).

[date36]
before dtype: object
after  dtype: datetime64[ns]
n different rows: 0
All values are identical (or both missing).

[menidt]
before dtype: object
after  dtype: datetime64[ns]
n different rows: 0
All values are identical (or both missing).

[pdaddt]
before dtype: object
after  dtype: datetime64[ns]
n different rows: 0
All values are identical (or both missing).

[iperrdt]
before dtype: object
after  dtype: datetime64[ns]
n different rows: 0
All values are identical (or both missing).

[bs

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\4179043236.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  knn_df[col] = after
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\4179043236.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  knn_df[col] = after
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\4179043236.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value 

In [16]:
# PPHN
knn_df['PPHN'] = ((knn_df['phudstdt'] - knn_df['birthdt']).dt.days <= 7).astype(int)
print(knn_df[['phudstdt', 'birthdt', 'PPHN']])


# PDA_7d_NTET
knn_df['PDA_7d_NTET'] = ((knn_df['PDA_TRD'] == 1) &
                          (knn_df['ntet'] == 2) &
                          ((knn_df['ntetdt'] - knn_df['pdaddt']).dt.days.between(0, 7))).astype(int)
# PDA_7d_IPERR
knn_df['PDA_7d_IPERR'] = ((knn_df['PDA_TRD'] == 1) &
                           (knn_df['iperr'].isin([2, 3, 4])) &
                           ((knn_df['iperrdt'] - knn_df['pdaddt']).dt.days.between(0, 7))).astype(int)# 7d_PDA
knn_df['7d_PDA'] = ((knn_df['PDA_7d_NTET'] == 1) | 
                     (knn_df['PDA_7d_IPERR'] == 1)).astype(int)
print(knn_df[['7d_PDA', 'PDA_7d_NTET', 'PDA_7d_IPERR']])

# SEPS_bef_IPERRorNTET : SEPS_bef_NTET OR SEPS_bef_IPERR
knn_df['PDA_7d_IPERRorNTET'] = (knn_df['PDA_7d_NTET'] | knn_df['PDA_7d_IPERR']
                               ).astype(int)
print(knn_df[['PDA_7d_IPERRorNTET']])


# death_label (Decease Label) & death_dt(Decease date)
knn_df['death_label'] = (knn_df['dcd'] == 3).astype(int)
knn_df['death_dt'] = knn_df['fdcdt'].where(knn_df['dcd'] == 3, pd.NaT)


# SEPS_bef_NTET : (ntet == 2 & bsf1 == 1) && bsfdt1 < ntetdt
knn_df['SEPS_bef_NTET'] = ((knn_df['ntet'] == 2) & 
                            (knn_df['bsf1'] == 1) & 
                            (knn_df['bsfdt1'] < knn_df['ntetdt'])).astype(int)
# SEPS_aft_NTET : (ntet == 2 & bsf1 == 1) && bsfdt1 > ntetdt
knn_df['SEPS_aft_NTET'] = ((knn_df['ntet'] == 2) & 
                            (knn_df['bsf1'] == 1) & 
                            (knn_df['bsfdt1'] > knn_df['ntetdt'])).astype(int)
# SEPS_bef_IPERR : (iperr == 2|3|4 & bsf1 == 1) && bsfdt1 < iperrdt
knn_df['SEPS_bef_IPERR'] = ((knn_df['iperr'].isin([2, 3, 4])) & 
                             (knn_df['bsf1'] == 1) & 
                             (knn_df['bsfdt1'] < knn_df['iperrdt'])).astype(int)
# SEPS_aft_IPERR : (iperr == 2|3|4 & bsf1 == 1) && bsfdt1 > iperrdt
knn_df['SEPS_aft_IPERR'] = ((knn_df['iperr'].isin([2, 3, 4])) & 
                             (knn_df['bsf1'] == 1) & 
                             (knn_df['bsfdt1'] > knn_df['iperrdt'])).astype(int)
print(knn_df[['SEPS_bef_NTET', 'SEPS_aft_NTET', 'SEPS_bef_IPERR', 'SEPS_aft_IPERR']])

# SEPS_bef_IPERRorNTET : SEPS_bef_NTET OR SEPS_bef_IPERR
knn_df['SEPS_bef_IPERRorNTET'] = (knn_df['SEPS_bef_NTET'] |
                                  knn_df['SEPS_bef_IPERR']).astype(int)
# SEPS_aft_IPERRorNTET : SEPS_aft_NTET OR SEPS_aft_IPERR
knn_df['SEPS_aft_IPERRorNTET'] = (knn_df['SEPS_aft_NTET'] |
                                  knn_df['SEPS_aft_IPERR']).astype(int)
print(knn_df[['SEPS_bef_IPERRorNTET', 'SEPS_aft_IPERRorNTET']])


# stday : week/day 형식 값 변환
knn_df['stday'] = knn_df['stday']
# NTET_day : (in ntet == 2)
knn_df['NTET_day'] = np.where(
    (knn_df['ntet'] == 2) & ((knn_df['ntetdt'] - knn_df['birthdt']).dt.days >= 0),
    (knn_df['ntetdt'] - knn_df['birthdt']).dt.days, np.nan)
# IPERR_day : (in iperr == 2|3|4)
knn_df['IPERR_day'] = np.where(
    (knn_df['iperr'].isin([2, 3, 4])) & ((knn_df['iperrdt'] - knn_df['birthdt']).dt.days >= 0),
    (knn_df['iperrdt'] - knn_df['birthdt']).dt.days, np.nan)
# EFYN_day : (in efyn == 2)
knn_df['EFYN_day'] = np.where(
    (knn_df['efyn'] == 2) & ((knn_df['efydt'] - knn_df['birthdt']).dt.days >= 0),
    (knn_df['efydt'] - knn_df['birthdt']).dt.days, np.nan)
# EFYN_day : NaN -> mean
efyn_na_count = knn_df['EFYN_day'].isna().sum()
efyn_total = len(knn_df)
efyn_na_ratio = (efyn_na_count / efyn_total) * 100
print(f"n of NaN -> Mean in EFYN_day: {efyn_na_count}")
print(f"{efyn_na_count} {efyn_na_ratio:.2f}% / {efyn_total}")
# knn_df['EFYN_day'] = knn_df['EFYN_day'].fillna(knn_df['EFYN_day'].mean())



# knn_df[['NTET_day', 'IPERR_day', 'EFYN_day']] = knn_df[['NTET_day'
#                                                         , 'IPERR_day'
#                                                         , 'EFYN_day']].fillna(0)
print(knn_df[['stday', 'NTET_day', 'IPERR_day', 'EFYN_day']])



# # stday : week/day 형식 값 변환
# knn_df['stday'] = knn_df['stday'] / 7
# # NTET_day : (in ntet == 2)
# knn_df['NTET_day'] = np.where(
#     (knn_df['ntet'] == 2) & ((knn_df['ntetdt'] - knn_df['birthdt']).dt.days >= 0),
#     (knn_df['ntetdt'] - knn_df['birthdt']).dt.days / 7, np.nan)
# # IPERR_day : (in iperr == 2|3|4)
# knn_df['IPERR_day'] = np.where(
#     (knn_df['iperr'].isin([2, 3, 4])) & ((knn_df['iperrdt'] - knn_df['birthdt']).dt.days >= 0),
#     (knn_df['iperrdt'] - knn_df['birthdt']).dt.days / 7, np.nan)
# # EFYN_day : (in efyn == 2)
# knn_df['EFYN_day'] = np.where(
#     (knn_df['efyn'] == 2) & ((knn_df['efydt'] - knn_df['birthdt']).dt.days >= 0),
#     (knn_df['efydt'] - knn_df['birthdt']).dt.days / 7, np.nan)

# knn_df[['NTET_day', 'IPERR_day', 'EFYN_day']] = knn_df[['NTET_day'
#                                                         , 'IPERR_day'
#                                                         , 'EFYN_day']].fillna(0)
# print(knn_df[['stday', 'NTET_day', 'IPERR_day', 'EFYN_day']])

# 불필요한 칼럼 삭제
knn_df.drop(columns=['acl', 'pdatr', 'pdad', 'pda_med-only', 'pda_op-only', 'pda_med+op'
                     , 'dcd', 'fdcdt', 'bsf1', 'bsfdt1'
                    ] + list(drop_cols)
            , inplace=True
           )

      phudstdt    birthdt  PPHN
0          NaT 2013-12-18     0
1          NaT 2013-11-15     0
2          NaT 2013-11-15     0
3          NaT 2013-11-15     0
4          NaT 2013-11-11     0
...        ...        ...   ...
20408      NaT 2022-11-28     0
20409      NaT 2022-12-27     0
20410      NaT 2022-12-01     0
20411      NaT 2022-11-30     0
20412      NaT 2022-11-30     0

[20352 rows x 3 columns]
       7d_PDA  PDA_7d_NTET  PDA_7d_IPERR
0           0            0             0
1           0            0             0
2           0            0             0
3           0            0             0
4           0            0             0
...       ...          ...           ...
20408       0            0             0
20409       0            0             0
20410       0            0             0
20411       0            0             0
20412       0            0             0

[20352 rows x 3 columns]
       PDA_7d_IPERRorNTET
0                       0
1                   

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\1302268995.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  knn_df['PPHN'] = ((knn_df['phudstdt'] - knn_df['birthdt']).dt.days <= 7).astype(int)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\1302268995.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  knn_df['PDA_7d_NTET'] = ((knn_df['PDA_TRD'] == 1) &
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\1302268995.py:11: SettingWithCopyWarning: 
A value is trying to

In [17]:
# '사망 원인' 칼럼 -> Binary column
death_mapping = {
    1: 'CR',
    2: 'NL',
    3: 'Inf',
    4: 'GI',
    5: 'CA',
    6: 'Otr'
}
new_columns = []
for code, abbrev in death_mapping.items():
    col_name = f'death_{abbrev}'
    knn_df[col_name] = (knn_df['death'] == code).astype(int)
    new_columns.append(col_name)
print("Created binary cols:")
for col in new_columns:
    print(f"\nCreated col_name: {col}")
    print(knn_df[col].value_counts(dropna=False))
    print("-" * 40)

knn_df.drop(columns='death', inplace=True)

Created binary cols:

Created col_name: death_CR
death_CR
0    19146
1     1206
Name: count, dtype: int64
----------------------------------------

Created col_name: death_NL
death_NL
0    20113
1      239
Name: count, dtype: int64
----------------------------------------

Created col_name: death_Inf
death_Inf
0    19934
1      418
Name: count, dtype: int64
----------------------------------------

Created col_name: death_GI
death_GI
0    20052
1      300
Name: count, dtype: int64
----------------------------------------

Created col_name: death_CA
death_CA
0    20258
1       94
Name: count, dtype: int64
----------------------------------------

Created col_name: death_Otr
death_Otr
0    20018
1      334
Name: count, dtype: int64
----------------------------------------


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\3740312812.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  knn_df[col_name] = (knn_df['death'] == code).astype(int)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\3740312812.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  knn_df[col_name] = (knn_df['death'] == code).astype(int)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\3740312812.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of 

In [19]:
analyse_dataframe(knn_df)

,NaN Counts,NaN Ratios (%),Unique Values Count
sex_sys_val,0,0.000000,2
birthdt,0,0.000000,3591
mulg,0,0.000000,4
bir1,0,0.000000,2
bir2,0,0.000000,2
bir3,0,0.000000,2
bir4,0,0.000000,2
prep,0,0.000000,2
dm,0,0.000000,3
htn,0,0.000000,3


In [21]:
knn_df

,sex_sys_val,birthdt,mulg,bir1,bir2,bir3,bir4,prep,dm,htn,...,SEPS_aft_IPERRorNTET,NTET_day,IPERR_day,EFYN_day,death_CR,death_NL,death_Inf,death_GI,death_CA,death_Otr
0,1,2013-12-18,1,0.0,0.0,0.0,0.0,1,1,2,...,0,NaN,NaN,10.0,0,0,0,0,0,0
1,1,2013-11-15,1,0.0,0.0,0.0,0.0,1,1,1,...,0,NaN,NaN,30.0,0,0,0,0,0,0
2,1,2013-11-15,2,1.0,0.0,0.0,0.0,1,1,1,...,0,NaN,NaN,14.0,0,0,0,0,0,0
3,1,2013-11-15,2,0.0,2.0,0.0,0.0,1,1,1,...,0,NaN,NaN,NaN,0,0,0,0,0,0
4,1,2013-11-11,1,0.0,0.0,0.0,0.0,1,1,1,...,0,NaN,NaN,31.0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20408,1,2022-11-28,1,0.0,0.0,0.0,0.0,1,1,1,...,0,NaN,NaN,27.0,0,0,0,0,0,0
20409,1,2022-12-27,1,0.0,0.0,0.0,0.0,1,1,1,...,0,NaN,NaN,34.0,0,0,0,0,0,0
20410,2,2022-12-01,1,0.0,0.0,0.0,0.0,2,1,1,...,0,NaN,NaN,5.0,0,0,0,0,0,0
20411,1,2022-11-30,2,0.0,2.0,0.0,0.0,2,2,1,...,0,NaN,NaN,9.0,0,0,0,0,0,0


In [22]:
for column in knn_df.columns:
    print(f"칼럼 이름: {column}")
    print(f"데이터 타입: {knn_df[column].dtype}")
    print(f"유니크한 값:")
    print(knn_df[column].unique())
    print("-" * 40)

칼럼 이름: sex_sys_val
데이터 타입: int64
유니크한 값:
[1 2]
----------------------------------------
칼럼 이름: birthdt
데이터 타입: datetime64[ns]
유니크한 값:
<DatetimeArray>
['2013-12-18 00:00:00', '2013-11-15 00:00:00', '2013-11-11 00:00:00',
 '2013-10-24 00:00:00', '2013-09-11 00:00:00', '2013-10-08 00:00:00',
 '2013-09-15 00:00:00', '2013-10-07 00:00:00', '2013-09-25 00:00:00',
 '2013-10-04 00:00:00',
 ...
 '2022-08-29 00:00:00', '2022-03-21 00:00:00', '2022-01-27 00:00:00',
 '2022-05-02 00:00:00', '2022-02-23 00:00:00', '2022-11-13 00:00:00',
 '2022-12-06 00:00:00', '2022-05-05 00:00:00', '2022-07-09 00:00:00',
 '2022-06-19 00:00:00']
Length: 3591, dtype: datetime64[ns]
----------------------------------------
칼럼 이름: mulg
데이터 타입: int64
유니크한 값:
[1 2 3 4]
----------------------------------------
칼럼 이름: bir1
데이터 타입: float64
유니크한 값:
[0. 1.]
----------------------------------------
칼럼 이름: bir2
데이터 타입: float64
유니크한 값:
[0. 2.]
----------------------------------------
칼럼 이름: bir3
데이터 타입: float64
유니크한 값:
[0. 3.]
-

In [23]:
knn_df.to_csv(os.path.join(crt_path, "knn_prep01_M07-Rev.csv"), encoding='utf-8-sig')

In [24]:
dfc = knn_df.copy()

In [25]:
dfc

,sex_sys_val,birthdt,mulg,bir1,bir2,bir3,bir4,prep,dm,htn,...,SEPS_aft_IPERRorNTET,NTET_day,IPERR_day,EFYN_day,death_CR,death_NL,death_Inf,death_GI,death_CA,death_Otr
0,1,2013-12-18,1,0.0,0.0,0.0,0.0,1,1,2,...,0,NaN,NaN,10.0,0,0,0,0,0,0
1,1,2013-11-15,1,0.0,0.0,0.0,0.0,1,1,1,...,0,NaN,NaN,30.0,0,0,0,0,0,0
2,1,2013-11-15,2,1.0,0.0,0.0,0.0,1,1,1,...,0,NaN,NaN,14.0,0,0,0,0,0,0
3,1,2013-11-15,2,0.0,2.0,0.0,0.0,1,1,1,...,0,NaN,NaN,NaN,0,0,0,0,0,0
4,1,2013-11-11,1,0.0,0.0,0.0,0.0,1,1,1,...,0,NaN,NaN,31.0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20408,1,2022-11-28,1,0.0,0.0,0.0,0.0,1,1,1,...,0,NaN,NaN,27.0,0,0,0,0,0,0
20409,1,2022-12-27,1,0.0,0.0,0.0,0.0,1,1,1,...,0,NaN,NaN,34.0,0,0,0,0,0,0
20410,2,2022-12-01,1,0.0,0.0,0.0,0.0,2,1,1,...,0,NaN,NaN,5.0,0,0,0,0,0,0
20411,1,2022-11-30,2,0.0,2.0,0.0,0.0,2,2,1,...,0,NaN,NaN,9.0,0,0,0,0,0,0


In [26]:
## 값 수정
dfc['sex_sys_val'] = dfc['sex_sys_val'].replace({2: 0})
dfc['bir2'] = dfc['bir2'].replace({2: 1})
dfc['bir3'] = dfc['bir3'].replace({3: 1})
dfc['bir4'] = dfc['bir4'].replace({4: 1})
# dfc['sterd2'] = dfc['sterd2'].replace({2: 1})
# dfc['sterd4'] = dfc['sterd4'].replace({4: 1})
# dfc['phud2'] = dfc['phud2'].replace({2: 1})
# dfc['phud3'] = dfc['phud3'].replace({3: 1})
# dfc['phud4'] = dfc['phud4'].replace({4: 1})
# dfc['phud5'] = dfc['phud5'].replace({5: 1})
# dfc['phud6'] = dfc['phud6'].replace({6: 1})
# dfc['strdut2'] = dfc['strdut2'].replace({2: 1})
# dfc['strdud2'] = dfc['strdud2'].replace({2: 1})
# dfc['strdud3'] = dfc['strdud3'].replace({3: 1})
# dfc['strdud4'] = dfc['strdud4'].replace({4: 1})
# dfc['paddr2'] = dfc['paddr2'].replace({2: 1})
# dfc['lbpd2'] = dfc['lbpd2'].replace({2: 1})
# dfc['lbpd3'] = dfc['lbpd3'].replace({3: 1})


# mcou, fcou 정확한 값이 아님
# dfc['mcou'] = dfc['mcou'].replace({1: "ROK"
#                                    , 2: "Vietnam"
#                                    , 3: "China"
#                                    , 4: "Philippines"
#                                    , 5: "Japan"
#                                    , 6: "Cambodia"
#                                    , 7: "USA"
#                                    , 8: "Thailand"
#                                    , 9: "Mongolia"
#                                    , 10: "Etc"})


## Unknown을 포함하는 칼럼 Binary화
columns_to_process = ['ph'
#                      , 'sterp', 'resuo', 'resup', 'resui', 'resuh', 'resue', 'resuc', 'rds'
#                      , 'strdut2', 'lbp'
                     ]
for col in columns_to_process:
    if col in dfc.columns:
        dfc[col] = dfc[col].replace({1: 0, 2: 1, 3: 'Unknown'})
        # 값 1에 대한 Binary 컬럼 생성
        dfc[f'{col}_1'] = (dfc[col] == 1).astype(int)
        # 값 'Unknown'에 대해 확인 후 Binary 컬럼 생성
        if 'Unknown' in dfc[col].values:
            dfc[f'{col}_Unknown'] = (dfc[col] == 'Unknown').astype(int)
        else:
            print(f"Value 'Unknown' not found in column '{col}'. Skipping creation of {col}_Unknown.")
        dfc.drop(columns=[col], inplace=True)


# 0 & 1 or+ 3 (Unknown) Binarise
columns_to_process = ['lbp', 'chor', 'ster', 'atbyn', 'resu', 'als', 'mph', 'sft', 'strdu'
                      , 'nese', 'ibif', 'meni', 'prom', 'ntet', 'eythtran'
                     ]
for col in columns_to_process:
    if col in dfc.columns:
        dfc[col] = dfc[col].replace({1: 0, 2: 1})
        has_3 = 3 in dfc[col].values
        dfc[f'{col}_1'] = (dfc[col] == 1).astype(int)
        dfc[f'{col}_0'] = (dfc[col] == 0).astype(int)
        if has_3:
            dfc[f'{col}_Unknown'] = (dfc[col] == 3).astype(int)
            print(f"Value '3' found in column '{col}'. Created '{col}_Unknown' column.")

        dfc.drop(columns=[col], inplace=True)


## cols for Binarise
# dfc['promn'] = dfc['promn'].replace({0: np.nan})
# dfc['sftup'] = dfc['sftup'].replace({0: np.nan})
# dfc['sftur'] = dfc['sftur'].replace({0: np.nan})
# dfc['sftth'] = dfc['sftth'].replace({0: np.nan})
# dfc['pdad'] = dfc['pdad'].replace({0: np.nan})
# dfc['pdadty'] = dfc['pdadty'].replace({0: np.nan})
# dfc['bsffs1'] = dfc['bsffs1'].replace({0: np.nan})
# dfc['bsffs2'] = dfc['bsffs2'].replace({0: np.nan})
# dfc['bsffs3'] = dfc['bsffs3'].replace({0: np.nan})
# dfc['fsffs1'] = dfc['fsffs1'].replace({0: np.nan})
# dfc['fsffs2'] = dfc['fsffs2'].replace({0: np.nan})
# dfc['menifs'] = dfc['menifs'].replace({0: np.nan})
dfc['ntetoy'] = dfc['ntetoy'].replace({0: np.nan})


# 'pmirnsg'
dfc['pmirnsg_0'] = dfc['pmirnsg'].isin([0]).astype(int)
dfc['pmirnsg_1+'] = dfc['pmirnsg'].isin([1, 2, 3, 4, 5]).astype(int)
print(dfc[['pmirnsg', 'pmirnsg_0', 'pmirnsg_1+']])
dfc.drop('pmirnsg', axis=1, inplace=True)

# 'pmirnsgnd'
dfc['pmirnsgnd'] = dfc.get('pmirnsgnd', 0).fillna(0).astype(int)

# 'efyn'
dfc['efyn_1'] = dfc['efyn'].isin([2]).astype(int)
print(dfc[['efyn', 'efyn_1']])
dfc.drop('efyn', axis=1, inplace=True)
dfc.rename(columns={'efyn_1': 'efyn'}, inplace=True)


## 'inhg'
dfc['inhg_0'] = dfc['inhg'].isin([1]).astype(int)
dfc['inhg_1'] = dfc['inhg'].isin([2, 3, 4, 5]).astype(int)
dfc['inhg_2+'] = dfc['inhg'].isin([3, 4, 5]).astype(int)
dfc['inhg_3+'] = dfc['inhg'].isin([4, 5]).astype(int)

dfc['ivh_0'] = dfc['inhg'].isin([1]).astype(int)
dfc['ivh_1'] = dfc['inhg'].isin([2]).astype(int)
dfc['ivh_2'] = dfc['inhg'].isin([3]).astype(int)
dfc['ivh_3'] = dfc['inhg'].isin([4]).astype(int)
dfc['ivh_4'] = dfc['inhg'].isin([5]).astype(int)
print(dfc[['inhg', 'inhg_1', 'inhg_2+', 'inhg_3+'
           , 'ivh_0', 'ivh_1', 'ivh_2', 'ivh_3', 'ivh_4']]
     )
dfc.drop('inhg', axis=1, inplace=True)

## 'bdp'
dfc['bdp_'] = dfc['bdp'].isin([2, 3, 4]).astype(int)
dfc['bdp_Moderate+'] = dfc['bdp'].isin([3, 4]).astype(int)
dfc.drop('bdp', axis=1, inplace=True)
dfc.rename(columns={'bdp_': 'bdp'}, inplace=True)
print(dfc[['bdp', 'bdp_Moderate+']])

## 'gagew', 'gaged'
dfc['ga'] = dfc['gagew'] + (dfc['gaged'] / 7)
dfc['ga'] = dfc['ga'].round(4)
print(dfc[['gagew', 'gaged', 'ga']])
dfc.drop(['gagew', 'gaged'], axis=1, inplace=True)

## 'mulg'
dfc['mulg_singleton'] = dfc['mulg'].isin([1]).astype(int)
dfc['mulg_multiple'] = dfc['mulg'].isin([2, 3, 4]).astype(int)
print(dfc[['mulg', 'mulg_singleton', 'mulg_multiple']])
dfc.drop('mulg', axis=1, inplace=True)

## 'dm'
dfc['dm_0'] = dfc['dm'].isin([1]).astype(int)
dfc['dm_1'] = dfc['dm'].isin([2, 3]).astype(int)
print(dfc[['dm', 'dm_0', 'dm_1']])
dfc.drop('dm', axis=1, inplace=True)

# 'delm'
dfc['delm_1'] = dfc['delm'].isin([1]).astype(int)
dfc['delm_2'] = dfc['delm'].isin([2]).astype(int)
print(dfc[['delm', 'delm_1', 'delm_2']])
dfc.drop('delm', axis=1, inplace=True)

## dth36dt : 출생일 + 교정(환산) 일수 = 교정 36주가 되는 날짜 계산
# dfc['dth36dt'] = dfc.apply(
#     lambda row: row['birthdt'] + pd.Timedelta((36 - row['ga']) * 7, unit='days')
#     , axis=1)
# print(dfc[['birthdt', 'ga', 'dth36dt']])

columns_to_process = ['prep', 'htn', 'amni', 'bpia'
                      , 'sftup', 'sftur', 'sftth', 'oxyt28', 'arre28', 'oxyt36', 'arre36'
                      , 'pvl', 'seps'
                      , 'ntety', 'ntetoy', 'iperr', 'iperroy'
#                      , 'pdad', 'pdadty', 'pdatr', 'acl'
#                      , 'bsffs1', 'bsffs2', 'bsffs3', 'fsffs1', 'fsffs2'
#                      , 'menifs'
#                      , 'mcou', 'fcou'    # >> 정확한 값이 아님.
                     ]

for col in columns_to_process:
    if col in dfc.columns:
        # 고유값을 기반으로 이진 컬럼 생성
        unique_values = dfc[col].dropna().unique()
        for value in unique_values:
            dfc[f'{col}_{int(value)}'] = (dfc[col] == value).astype(int)
        # 생성된 컬럼 이름 출력
        cols = [new_col for new_col in dfc.columns if new_col.startswith(col)]
        print(f"Generated columns for '{col}': {cols}")
        # 원래 칼럼 삭제
        dfc.drop(columns=[col], inplace=True)

dfc



# # 'chor' : 1, 'Unknown'을 기준으로 Binary 화
# dfc['chor'] = dfc['chor'].replace({1:0, 2:1, 3: 'Unknown'})
# if 'chor' in dfc.columns:
#     # chor_1: 값이 1인 경우 1, 그 외는 0
#     dfc['chor_1'] = (dfc['chor'] == 1).astype(int)
#     # chor_unknown: 값이 'Unknown'인 경우 1, 그 외는 0
#     dfc['chor_Unknown'] = (dfc['chor'] == 'Unknown').astype(int)
# print(dfc[['chor', 'chor_1', 'chor_Unknown']])
# dfc.drop(columns=['chor'], inplace=True)

Value 'Unknown' not found in column 'ph'. Skipping creation of ph_Unknown.
Value '3' found in column 'chor'. Created 'chor_Unknown' column.
Value '3' found in column 'ster'. Created 'ster_Unknown' column.
Value '3' found in column 'atbyn'. Created 'atbyn_Unknown' column.
Value '3' found in column 'resu'. Created 'resu_Unknown' column.
Value '3' found in column 'prom'. Created 'prom_Unknown' column.
Value '3' found in column 'ntet'. Created 'ntet_Unknown' column.
       pmirnsg  pmirnsg_0  pmirnsg_1+
0          0.0          1           0
1          2.0          0           1
2          0.0          1           0
3          0.0          1           0
4          1.0          0           1
...        ...        ...         ...
20408      0.0          1           0
20409      2.0          0           1
20410      0.0          1           0
20411      0.0          1           0
20412      0.0          1           0

[20352 rows x 3 columns]
       efyn  efyn_1
0         2       1
1         2

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\2756643273.py:170: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dfc[f'{col}_{int(value)}'] = (dfc[col] == value).astype(int)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\2756643273.py:170: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dfc[f'{col}_{int(value)}'] = (dfc[col] == value).astype(int)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37432\2756643273.py:170: PerformanceWarning: DataFrame is highly fragmented.  This is usually the resu

,sex_sys_val,birthdt,bir1,bir2,bir3,bir4,bird,admd,apgs1,apgs5,...,ntetoy_3,ntetoy_1,iperr_1,iperr_2,iperr_3,iperr_4,iperroy_1,iperroy_4,iperroy_3,iperroy_2
0,1,2013-12-18,0.0,0.0,0.0,0.0,2013-12-18,2013-12-18,4.0,5.0,...,0,0,1,0,0,0,1,0,0,0
1,1,2013-11-15,0.0,0.0,0.0,0.0,2013-11-15,2013-11-15,4.0,5.0,...,0,0,1,0,0,0,1,0,0,0
2,1,2013-11-15,1.0,0.0,0.0,0.0,2013-11-15,2013-11-15,5.0,6.0,...,0,0,1,0,0,0,1,0,0,0
3,1,2013-11-15,0.0,1.0,0.0,0.0,2013-11-15,2013-11-15,6.0,7.0,...,0,0,1,0,0,0,1,0,0,0
4,1,2013-11-11,0.0,0.0,0.0,0.0,2013-11-11,2013-11-11,4.0,5.0,...,0,0,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20408,1,2022-11-28,0.0,0.0,0.0,0.0,2022-11-28,2022-11-28,5.0,8.0,...,0,0,1,0,0,0,1,0,0,0
20409,1,2022-12-27,0.0,0.0,0.0,0.0,2022-12-27,2022-12-27,4.0,7.0,...,0,0,1,0,0,0,1,0,0,0
20410,0,2022-12-01,0.0,0.0,0.0,0.0,2022-12-01,2022-12-01,5.0,7.0,...,0,0,1,0,0,0,1,0,0,0
20411,1,2022-11-30,0.0,1.0,0.0,0.0,2022-11-30,2022-11-30,3.0,7.0,...,0,0,1,0,0,0,1,0,0,0


In [28]:
analyse_dataframe(dfc)

,NaN Counts,NaN Ratios (%),Unique Values Count
sex_sys_val,0,0.000000,2
birthdt,0,0.000000,3591
bir1,0,0.000000,2
bir2,0,0.000000,2
bir3,0,0.000000,2
bir4,0,0.000000,2
bird,0,0.000000,3591
admd,0,0.000000,3592
apgs1,121,0.594536,11
apgs5,117,0.574882,11


In [31]:
df = dfc.copy()

print(len(df.columns), '\n')

# 1) drop_cols
drop_cols = [
    'ga',
    'bir1', 'bir2', 'bir3', 'bir4',
    'death36'
]

datetime_cols = [
    'ntetdt', 'ntetdty', 'date362', 'date36', 'menidt', 'pdaddth', 'pdaddt',
    'birthdt', 'admd', 'birtm', 'bird', 'efydt', 'death_dt', 'iperrdt'
]
drop_cols += datetime_cols

rsndth_cols = ['death_CR', 'death_NL', 'death_Inf', 'death_GI', 'death_CA', 'death_Otr']
drop_cols += rsndth_cols

drop_cols = [c for c in drop_cols if c in df.columns]

print('len(drop_cols): ', len(drop_cols))
print("drop_cols:", drop_cols)

# 2) target_keys -> tgt_cols
target_keys = [
    'ntet_', 'ntety_', 'ntetoy_',
    'SEPS_bef_NTET', 'SEPS_aft_NTET',
    'NTET_day', 'PDA_7d_NTET',
    'invfpod',
    'iperr_', 'iperroy_',
    'SEPS_bef_IPERR', 'SEPS_aft_IPERR',
    'IPERR_day', 'PDA_7d_IPERR', 'PDA_7d_IPERRorNTET',
    'SEPS_bef_IPERRorNTET', 'SEPS_aft_IPERRorNTET',
    '7d_PDA',
    'bdp', 'bdp_Moderate+', 'strdu_',
    'inhg_', 'pvl_', 'nese_',
    'seps_3',
    'pmirnsg_', 'pmirnsgnd',
    'stday',
    'oxyt36_', 'arre36_',
    'bwei',
    'btem', 'bhei', 'bhead',
    'ivh_'
]

tgt_cols = [
    col for col in df.columns
    if col in target_keys or any(col.startswith(prefix) for prefix in target_keys)
]

print('len(tgt_cols): ', len(tgt_cols))
print("tgt_cols:", tgt_cols)

# 3) clustering 포함 여부 계산
excluded_by_rule = set(drop_cols)
excluded_by_target = set(tgt_cols)
excluded_outcome = {'death_label'} if 'death_label' in df.columns else set()

clustering_cols = [
    col for col in df.columns
    if col not in excluded_by_rule
    and col not in excluded_by_target
    and col not in excluded_outcome
]

print('len(clustering_cols): ', len(clustering_cols))
print("clustering_cols:", clustering_cols)

# 4) Full imputation table 생성 (모든 변수 포함)
rows = []

for col in df.columns:
    s = df[col]
    nonmiss = s.dropna()
    nunique = nonmiss.nunique()

    # 값 요약: 범주 수가 적은 경우만 표시
    value_summary = ""
    if nunique <= 10:
        vc = nonmiss.value_counts(dropna=False).sort_index()
        value_summary = ", ".join([f"{k}: {v}" for k, v in vc.items()])

    if col in clustering_cols:
        used_for_clustering = 'Yes'
        reason = 'Included'
    elif col in excluded_outcome:
        used_for_clustering = 'No'
        reason = 'Outcome excluded'
    elif col in excluded_by_target:
        used_for_clustering = 'No'
        reason = 'Target/derived excluded'
    elif col in excluded_by_rule:
        used_for_clustering = 'No'
        reason = 'Dropped by predefined rule'
    else:
        used_for_clustering = 'No'
        reason = 'Not in final feature set'

    rows.append({
        'Variable': col,
        'Missing_n': int(s.isna().sum()),
        'Missing_pct': float(s.isna().mean() * 100),
        'Nonmissing_n': int(s.notna().sum()),
        'dtype': str(s.dtype),
        'Unique_nonmissing_n': int(nunique),
        'Value_summary': value_summary,
        'Used_for_clustering': used_for_clustering,
        'Clustering_flag_reason': reason
    })

full_imputation_table = pd.DataFrame(rows)

# 보기 좋게 정렬
full_imputation_table = full_imputation_table.sort_values(
    by=['Used_for_clustering', 'Missing_pct', 'Missing_n', 'Variable'],
    ascending=[False, False, False, True]
).reset_index(drop=True)

print(full_imputation_table.shape)
full_imputation_table

172 

len(drop_cols):  26
drop_cols: ['ga', 'bir1', 'bir2', 'bir3', 'bir4', 'death36', 'ntetdt', 'ntetdty', 'date362', 'date36', 'menidt', 'pdaddth', 'pdaddt', 'birthdt', 'admd', 'birtm', 'bird', 'efydt', 'death_dt', 'iperrdt', 'death_CR', 'death_NL', 'death_Inf', 'death_GI', 'death_CA', 'death_Otr']
len(tgt_cols):  67
tgt_cols: ['bwei', 'bhei', 'bheiuk', 'bhead', 'bheaduk', 'btem', 'btemuk', 'invfpod', 'pmirnsgnd', 'stday', 'PDA_7d_NTET', 'PDA_7d_IPERR', '7d_PDA', 'PDA_7d_IPERRorNTET', 'SEPS_bef_NTET', 'SEPS_aft_NTET', 'SEPS_bef_IPERR', 'SEPS_aft_IPERR', 'SEPS_bef_IPERRorNTET', 'SEPS_aft_IPERRorNTET', 'NTET_day', 'IPERR_day', 'strdu_1', 'strdu_0', 'nese_1', 'nese_0', 'ntet_1', 'ntet_0', 'ntet_Unknown', 'pmirnsg_0', 'pmirnsg_1+', 'inhg_0', 'inhg_1', 'inhg_2+', 'inhg_3+', 'ivh_0', 'ivh_1', 'ivh_2', 'ivh_3', 'ivh_4', 'bdp', 'bdp_Moderate+', 'oxyt36_1', 'oxyt36_2', 'oxyt36_3', 'arre36_0', 'arre36_6', 'arre36_7', 'pvl_1', 'pvl_3', 'pvl_2', 'seps_3', 'ntety_1', 'ntety_2', 'ntety_3', 'ntety_

,Variable,Missing_n,Missing_pct,Nonmissing_n,dtype,Unique_nonmissing_n,Value_summary,Used_for_clustering,Clustering_flag_reason
0,bbph,4597,22.587461,15755,float64,114,,Yes,Included
1,EFYN_day,2527,12.416470,17825,float64,169,,Yes,Included
2,apgs1,121,0.594536,20231,float64,11,,Yes,Included
3,apgs5,117,0.574882,20235,float64,11,,Yes,Included
4,PDA_TRD,0,0.000000,20352,int32,2,"0: 13642, 1: 6710",Yes,Included
...,...,...,...,...,...,...,...,...,...
167,pvl_3,0,0.000000,20352,int32,2,"0: 19618, 1: 734",No,Target/derived excluded
168,seps_3,0,0.000000,20352,int32,2,"0: 20270, 1: 82",No,Target/derived excluded
169,stday,0,0.000000,20352,int64,313,,No,Target/derived excluded
170,strdu_0,0,0.000000,20352,int32,2,"0: 4702, 1: 15650",No,Target/derived excluded


In [32]:
# SAVE :: Full missing table

full_imputation_table.to_csv(
    'full_imputation_table_with_clustering_flag.csv',
    index=True,
    encoding='utf-8-sig'
)

In [ ]:
# 1. Original method (Mean Imp)
dfc_1 = dfc.copy
mean_cols = ['apgs1', 'apgs5', 'btem', 'bbph', 'bhead', 'bhei']
dfc[mean_cols] = knn_df[mean_cols].fillna(knn_df[mean_cols].mean())

In [68]:
analyse_dataframe(dfc)

,NaN Counts,NaN Ratios (%),Unique Values Count
sex_sys_val,0,0.000000,2
bir1,0,0.000000,2
bir2,0,0.000000,2
bir3,0,0.000000,2
bir4,0,0.000000,2
birdh,0,0.000000,24
birdm,0,0.000000,60
apgs1,121,0.594536,11
apguk1,20231,99.405464,1
apgs5,117,0.574882,11


In [33]:
for column in dfc.columns:
    print(f"칼럼 이름: {column}")
    print(f"데이터 타입: {dfc[column].dtype}")
    print(f"유니크한 값:")
    print(dfc[column].unique())
    print("-" * 40)

칼럼 이름: sex_sys_val
데이터 타입: int64
유니크한 값:
[1 0]
----------------------------------------
칼럼 이름: birthdt
데이터 타입: datetime64[ns]
유니크한 값:
<DatetimeArray>
['2013-12-18 00:00:00', '2013-11-15 00:00:00', '2013-11-11 00:00:00',
 '2013-10-24 00:00:00', '2013-09-11 00:00:00', '2013-10-08 00:00:00',
 '2013-09-15 00:00:00', '2013-10-07 00:00:00', '2013-09-25 00:00:00',
 '2013-10-04 00:00:00',
 ...
 '2022-08-29 00:00:00', '2022-03-21 00:00:00', '2022-01-27 00:00:00',
 '2022-05-02 00:00:00', '2022-02-23 00:00:00', '2022-11-13 00:00:00',
 '2022-12-06 00:00:00', '2022-05-05 00:00:00', '2022-07-09 00:00:00',
 '2022-06-19 00:00:00']
Length: 3591, dtype: datetime64[ns]
----------------------------------------
칼럼 이름: bir1
데이터 타입: float64
유니크한 값:
[0. 1.]
----------------------------------------
칼럼 이름: bir2
데이터 타입: float64
유니크한 값:
[0. 1.]
----------------------------------------
칼럼 이름: bir3
데이터 타입: float64
유니크한 값:
[0. 1.]
----------------------------------------
칼럼 이름: bir4
데이터 타입: float64
유니크한 값:
[0. 1.]
-

In [44]:
dfc

,sex_sys_val,birthdt,bir1,bir2,bir3,bir4,bird,admd,apgs1,apgs5,...,ntetoy_3,ntetoy_1,iperr_1,iperr_2,iperr_3,iperr_4,iperroy_1,iperroy_4,iperroy_3,iperroy_2
0,1,2013-12-18,0.0,0.0,0.0,0.0,2013-12-18,2013-12-18,4.0,5.0,...,0,0,1,0,0,0,1,0,0,0
1,1,2013-11-15,0.0,0.0,0.0,0.0,2013-11-15,2013-11-15,4.0,5.0,...,0,0,1,0,0,0,1,0,0,0
2,1,2013-11-15,1.0,0.0,0.0,0.0,2013-11-15,2013-11-15,5.0,6.0,...,0,0,1,0,0,0,1,0,0,0
3,1,2013-11-15,0.0,1.0,0.0,0.0,2013-11-15,2013-11-15,6.0,7.0,...,0,0,1,0,0,0,1,0,0,0
4,1,2013-11-11,0.0,0.0,0.0,0.0,2013-11-11,2013-11-11,4.0,5.0,...,0,0,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20408,1,2022-11-28,0.0,0.0,0.0,0.0,2022-11-28,2022-11-28,5.0,8.0,...,0,0,1,0,0,0,1,0,0,0
20409,1,2022-12-27,0.0,0.0,0.0,0.0,2022-12-27,2022-12-27,4.0,7.0,...,0,0,1,0,0,0,1,0,0,0
20410,0,2022-12-01,0.0,0.0,0.0,0.0,2022-12-01,2022-12-01,5.0,7.0,...,0,0,1,0,0,0,1,0,0,0
20411,1,2022-11-30,0.0,1.0,0.0,0.0,2022-11-30,2022-11-30,3.0,7.0,...,0,0,1,0,0,0,1,0,0,0


In [46]:
out_dir = os.path.join(f"{base}/PedRes_Revision/dataset")
os.makedirs(out_dir, exist_ok=True)

# imputation 대상 칼럼
mean_cols = ['apgs1', 'apgs5', 'btem', 'bbph', 'bhead', 'bhei']
imp_cols = [c for c in mean_cols if c in dfc.columns]

print("Imputation columns:", imp_cols)

# 1) Non-imputation 상태 저장
dfc_nonimp = dfc.copy()
dfc_nonimp.to_csv(
    os.path.join(out_dir, 'knn_prep-fin_M08-Rev_NonImp.csv'),
    encoding='utf-8-sig'
)

# 2) Original 방식: mean imputation
dfc1 = dfc.copy()
dfc1[imp_cols] = dfc1[imp_cols].fillna(dfc1[imp_cols].mean())
dfc1.to_csv(
    os.path.join(out_dir, 'knn_prep-fin_M08-Rev_Orig.csv'),
    encoding='utf-8-sig'
)

# 3) Median imputation
dfc2 = dfc.copy()
dfc2[imp_cols] = dfc2[imp_cols].fillna(dfc2[imp_cols].median())
dfc2.to_csv(
    os.path.join(out_dir, 'knn_prep-fin_M08-Rev_MedImp.csv'),
    encoding='utf-8-sig'
)

# 확인
print("\nSaved files:")
print(os.path.join(out_dir, 'knn_prep-fin_M08-Rev_NonImp.csv'))
print(os.path.join(out_dir, 'knn_prep-fin_M08-Rev_Orig.csv'))
print(os.path.join(out_dir, 'knn_prep-fin_M08-Rev_MedImp.csv'))

print("\nRemaining NaNs after each save target:")
print("NonImp")
print(dfc_nonimp[imp_cols].isna().sum())
print("\nOrig (mean): NaN values")
print(dfc1[imp_cols].isna().sum())
print("\nMedImp (median): NaN values")
print(dfc2[imp_cols].isna().sum())

Imputation columns: ['apgs1', 'apgs5', 'btem', 'bbph', 'bhead', 'bhei']

Saved files:
C:\Dev_AIWM\jupyter\KNN/PedRes_Revision/dataset\knn_prep-fin_M08-Rev_NonImp.csv
C:\Dev_AIWM\jupyter\KNN/PedRes_Revision/dataset\knn_prep-fin_M08-Rev_Orig.csv
C:\Dev_AIWM\jupyter\KNN/PedRes_Revision/dataset\knn_prep-fin_M08-Rev_MedImp.csv

Remaining NaNs after each save target:
NonImp
apgs1     121
apgs5     117
btem      631
bbph     4597
bhead    1230
bhei     1137
dtype: int64

Orig (mean): NaN values
apgs1    0
apgs5    0
btem     0
bbph     0
bhead    0
bhei     0
dtype: int64

MedImp (median): NaN values
apgs1    0
apgs5    0
btem     0
bbph     0
bhead    0
bhei     0
dtype: int64


In [47]:
dfc

,sex_sys_val,birthdt,bir1,bir2,bir3,bir4,bird,admd,apgs1,apgs5,...,ntetoy_3,ntetoy_1,iperr_1,iperr_2,iperr_3,iperr_4,iperroy_1,iperroy_4,iperroy_3,iperroy_2
0,1,2013-12-18,0.0,0.0,0.0,0.0,2013-12-18,2013-12-18,4.0,5.0,...,0,0,1,0,0,0,1,0,0,0
1,1,2013-11-15,0.0,0.0,0.0,0.0,2013-11-15,2013-11-15,4.0,5.0,...,0,0,1,0,0,0,1,0,0,0
2,1,2013-11-15,1.0,0.0,0.0,0.0,2013-11-15,2013-11-15,5.0,6.0,...,0,0,1,0,0,0,1,0,0,0
3,1,2013-11-15,0.0,1.0,0.0,0.0,2013-11-15,2013-11-15,6.0,7.0,...,0,0,1,0,0,0,1,0,0,0
4,1,2013-11-11,0.0,0.0,0.0,0.0,2013-11-11,2013-11-11,4.0,5.0,...,0,0,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20408,1,2022-11-28,0.0,0.0,0.0,0.0,2022-11-28,2022-11-28,5.0,8.0,...,0,0,1,0,0,0,1,0,0,0
20409,1,2022-12-27,0.0,0.0,0.0,0.0,2022-12-27,2022-12-27,4.0,7.0,...,0,0,1,0,0,0,1,0,0,0
20410,0,2022-12-01,0.0,0.0,0.0,0.0,2022-12-01,2022-12-01,5.0,7.0,...,0,0,1,0,0,0,1,0,0,0
20411,1,2022-11-30,0.0,1.0,0.0,0.0,2022-11-30,2022-11-30,3.0,7.0,...,0,0,1,0,0,0,1,0,0,0


In [52]:
## 4) MICE / Iterative imputation

# i=20
dfc3 = dfc.copy()

mice_imputer = IterativeImputer(
    random_state=64,
    max_iter=20,
    sample_posterior=False,
    initial_strategy='mean'
)

dfc3[imp_cols] = mice_imputer.fit_transform(dfc3[imp_cols])

dfc3.to_csv(
    os.path.join(out_dir, 'knn_prep-fin_M08-Rev_MICEImp-I20.csv'),
    encoding='utf-8-sig'
)

print("\nSaved files:")
print(os.path.join(out_dir, 'knn_prep-fin_M08-Rev_MICEImp-I20.csv'))
print("\nMICE: NaN values")
print(dfc3[imp_cols].isna().sum())


# i=10
dfc3 = dfc.copy()

mice_imputer = IterativeImputer(
    random_state=64,
    max_iter=10,
    sample_posterior=False,
    initial_strategy='mean'
)

dfc3[imp_cols] = mice_imputer.fit_transform(dfc3[imp_cols])

dfc3.to_csv(
    os.path.join(out_dir, 'knn_prep-fin_M08-Rev_MICEImp-I10.csv'),
    encoding='utf-8-sig'
)

print("\nSaved files:")
print(os.path.join(out_dir, 'knn_prep-fin_M08-Rev_MICEImp-I10.csv'))
print("\nMICE: NaN values")
print(dfc3[imp_cols].isna().sum())


Saved files:
C:\Dev_AIWM\jupyter\KNN/PedRes_Revision/dataset\knn_prep-fin_M08-Rev_MICEImp-I20.csv

MICE: NaN values
apgs1    0
apgs5    0
btem     0
bbph     0
bhead    0
bhei     0
dtype: int64


C:\Users\AIWM_W01\.conda\envs\Dev_AIWM\lib\site-packages\sklearn\impute\_iterative.py:801: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(



Saved files:
C:\Dev_AIWM\jupyter\KNN/PedRes_Revision/dataset\knn_prep-fin_M08-Rev_MICEImp-I10.csv

MICE: NaN values
apgs1    0
apgs5    0
btem     0
bbph     0
bhead    0
bhei     0
dtype: int64


In [54]:
## 5) KNN imputation

# K=5
dfc4 = dfc.copy()

knn_imputer = KNNImputer(
    n_neighbors=5,
    weights='distance'
)

dfc4[imp_cols] = knn_imputer.fit_transform(dfc4[imp_cols])

dfc4.to_csv(
    os.path.join(out_dir, 'knn_prep-fin_M08-Rev_KNNImp-K5.csv'),
    encoding='utf-8-sig'
)

# 확인
print("\nSaved files:")
print(os.path.join(out_dir, 'knn_prep-fin_M08-Rev_KNNImp-K5.csv'))
print("\nKNN: NaN values")
print(dfc4[imp_cols].isna().sum())


# K=10
dfc4 = dfc.copy()

knn_imputer = KNNImputer(
    n_neighbors=10,
    weights='distance'
)

dfc4[imp_cols] = knn_imputer.fit_transform(dfc4[imp_cols])

dfc4.to_csv(
    os.path.join(out_dir, 'knn_prep-fin_M08-Rev_KNNImp-K10.csv'),
    encoding='utf-8-sig'
)

# 확인
print("\nSaved files:")
print(os.path.join(out_dir, 'knn_prep-fin_M08-Rev_KNNImp-K10.csv'))
print("\nKNN: NaN values")
print(dfc4[imp_cols].isna().sum())

# Non-weights
# K=10
dfc4 = dfc.copy()

knn_imputer = KNNImputer(
    n_neighbors=10,
)

dfc4[imp_cols] = knn_imputer.fit_transform(dfc4[imp_cols])

dfc4.to_csv(
    os.path.join(out_dir, 'knn_prep-fin_M08-Rev_KNNImp-K10NW.csv'),
    encoding='utf-8-sig'
)

# 확인
print("\nSaved files:")
print(os.path.join(out_dir, 'knn_prep-fin_M08-Rev_KNNImp-K10NW.csv'))
print("\nKNN: NaN values")
print(dfc4[imp_cols].isna().sum())


Saved files:
C:\Dev_AIWM\jupyter\KNN/PedRes_Revision/dataset\knn_prep-fin_M08-Rev_KNNImp-K5.csv

KNN: NaN values
apgs1    0
apgs5    0
btem     0
bbph     0
bhead    0
bhei     0
dtype: int64

Saved files:
C:\Dev_AIWM\jupyter\KNN/PedRes_Revision/dataset\knn_prep-fin_M08-Rev_KNNImp-K10.csv

KNN: NaN values
apgs1    0
apgs5    0
btem     0
bbph     0
bhead    0
bhei     0
dtype: int64

Saved files:
C:\Dev_AIWM\jupyter\KNN/PedRes_Revision/dataset\knn_prep-fin_M08-Rev_KNNImp-K10NW.csv

KNN: NaN values
apgs1    0
apgs5    0
btem     0
bbph     0
bhead    0
bhei     0
dtype: int64


In [55]:
# K=20
dfc5 = dfc.copy()

knn_imputer = KNNImputer(
    n_neighbors=20,
    weights='distance'
)

dfc5[imp_cols] = knn_imputer.fit_transform(dfc5[imp_cols])

dfc5.to_csv(
    os.path.join(out_dir, 'knn_prep-fin_M08-Rev_KNNImp-K20.csv'),
    encoding='utf-8-sig'
)

# 확인
print("\nSaved files:")
print(os.path.join(out_dir, 'knn_prep-fin_M08-Rev_KNNImp-K20.csv'))
print("\nKNN: NaN values")
print(dfc5[imp_cols].isna().sum())


Saved files:
C:\Dev_AIWM\jupyter\KNN/PedRes_Revision/dataset\knn_prep-fin_M08-Rev_KNNImp-K20.csv

KNN: NaN values
apgs1    0
apgs5    0
btem     0
bbph     0
bhead    0
bhei     0
dtype: int64


In [20]:
# dfc.to_csv(os.path.join(crt_path, "knn_prep-fin_M08-Rev.csv"), encoding='utf-8-sig')

In [43]:
dfc_nonimp

,sex_sys_val,birthdt,bir1,bir2,bir3,bir4,bird,admd,apgs1,apgs5,...,ntetoy_3,ntetoy_1,iperr_1,iperr_2,iperr_3,iperr_4,iperroy_1,iperroy_4,iperroy_3,iperroy_2
0,1,2013-12-18,0.0,0.0,0.0,0.0,2013-12-18,2013-12-18,4.0,5.0,...,0,0,1,0,0,0,1,0,0,0
1,1,2013-11-15,0.0,0.0,0.0,0.0,2013-11-15,2013-11-15,4.0,5.0,...,0,0,1,0,0,0,1,0,0,0
2,1,2013-11-15,1.0,0.0,0.0,0.0,2013-11-15,2013-11-15,5.0,6.0,...,0,0,1,0,0,0,1,0,0,0
3,1,2013-11-15,0.0,1.0,0.0,0.0,2013-11-15,2013-11-15,6.0,7.0,...,0,0,1,0,0,0,1,0,0,0
4,1,2013-11-11,0.0,0.0,0.0,0.0,2013-11-11,2013-11-11,4.0,5.0,...,0,0,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20408,1,2022-11-28,0.0,0.0,0.0,0.0,2022-11-28,2022-11-28,5.0,8.0,...,0,0,1,0,0,0,1,0,0,0
20409,1,2022-12-27,0.0,0.0,0.0,0.0,2022-12-27,2022-12-27,4.0,7.0,...,0,0,1,0,0,0,1,0,0,0
20410,0,2022-12-01,0.0,0.0,0.0,0.0,2022-12-01,2022-12-01,5.0,7.0,...,0,0,1,0,0,0,1,0,0,0
20411,1,2022-11-30,0.0,1.0,0.0,0.0,2022-11-30,2022-11-30,3.0,7.0,...,0,0,1,0,0,0,1,0,0,0


In [22]:
# # 1) 생존자( death36 == 0 )는 조건 없이 모두 유지
dfc_alive = dfc[dfc['death_label'] == 0]
dfc_alive.drop(columns=['death36', 'bir1', 'bir2', 'bir3', 'bir4'], inplace=True)
print(dfc_alive)

# dfc_dead = dfc[dfc['death_label'] == 1]

# # 2) 사망자( death36 == 1 ) 중에서
# #    dth36dt > ntetdt OR dth36dt > iperrdt 인 경우만 유지
# total_death_label_1 = dfc['death_label'].eq(1).sum()
# print(f"Total individuals with death_label = 1: {total_death_label_1}")
# before_filtering = len(dfc)
# dfc_dead = dfc[
#     (dfc['death_label'] == 1) &
#     ((dfc['death_dt'] > dfc['ntetdt']) | (dfc['death_dt'] > dfc['iperrdt']))
# ]
# after_filtering = len(dfc_dead)
# removed_count = total_death_label_1 - after_filtering
# print(f"Remaining individuals after filtering: {after_filtering}")
# print(f"Removed individuals: {removed_count}")

# # 3) 두 데이터프레임을 합쳐 최종 데이터프레임 생성
# dfc_f = pd.concat([dfc_alive, dfc_dead], ignore_index=True)

# dfc_f.drop(columns=['death36', 'bir1', 'bir2', 'bir3', 'bir4'], inplace=True)
# dfc_f

dfc.drop(columns=['death36', 'bir1', 'bir2', 'bir3', 'bir4'], inplace=True)
dfc

       sex_sys_val     birthdt        bird        admd  apgs1  apgs5  bwei  \
0                1  2013-12-18  2013-12-18  2013-12-18    4.0    5.0  1350   
1                1  2013-11-15  2013-11-15  2013-11-15    4.0    5.0   790   
2                1  2013-11-15  2013-11-15  2013-11-15    5.0    6.0  1400   
3                1  2013-11-15  2013-11-15  2013-11-15    6.0    7.0  1340   
4                1  2013-11-11  2013-11-11  2013-11-11    4.0    5.0  1240   
...            ...         ...         ...         ...    ...    ...   ...   
20408            1  2022-11-28  2022-11-28  2022-11-28    5.0    8.0  1350   
20409            1  2022-12-27  2022-12-27  2022-12-27    4.0    7.0   860   
20410            0  2022-12-01  2022-12-01  2022-12-01    5.0    7.0  1910   
20411            1  2022-11-30  2022-11-30  2022-11-30    3.0    7.0  1660   
20412            1  2022-11-30  2022-11-30  2022-11-30    4.0    6.0  1740   

       mage  gran  parn  ...  ntetoy_3  ntetoy_1  iperr_1  iper

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_27716\248818792.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfc_alive.drop(columns=['death36', 'bir1', 'bir2', 'bir3', 'bir4'], inplace=True)


,sex_sys_val,birthdt,bird,admd,apgs1,apgs5,bwei,mage,gran,parn,...,ntetoy_3,ntetoy_1,iperr_1,iperr_2,iperr_3,iperr_4,iperroy_1,iperroy_4,iperroy_3,iperroy_2
0,1,2013-12-18,2013-12-18,2013-12-18,4.0,5.0,1350,29,1,0,...,0,0,1,0,0,0,1,0,0,0
1,1,2013-11-15,2013-11-15,2013-11-15,4.0,5.0,790,35,1,0,...,0,0,1,0,0,0,1,0,0,0
2,1,2013-11-15,2013-11-15,2013-11-15,5.0,6.0,1400,26,1,0,...,0,0,1,0,0,0,1,0,0,0
3,1,2013-11-15,2013-11-15,2013-11-15,6.0,7.0,1340,26,1,0,...,0,0,1,0,0,0,1,0,0,0
4,1,2013-11-11,2013-11-11,2013-11-11,4.0,5.0,1240,36,4,1,...,0,0,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20408,1,2022-11-28,2022-11-28,2022-11-28,5.0,8.0,1350,23,1,0,...,0,0,1,0,0,0,1,0,0,0
20409,1,2022-12-27,2022-12-27,2022-12-27,4.0,7.0,860,38,3,1,...,0,0,1,0,0,0,1,0,0,0
20410,0,2022-12-01,2022-12-01,2022-12-01,5.0,7.0,1910,30,2,0,...,0,0,1,0,0,0,1,0,0,0
20411,1,2022-11-30,2022-11-30,2022-11-30,3.0,7.0,1660,34,1,0,...,0,0,1,0,0,0,1,0,0,0


In [23]:
# # 생후 7일 이내 사망 제외
# mask = (dfc_f['death_dt'] - dfc_f['birthdt']).dt.days <= 7

# removed_count = mask.sum()
# dfc_f = dfc_f[~mask].reset_index(drop=True)
# print(f'Removed n: {removed_count}')

# dfc_f

In [24]:
dfc_alive.to_csv(os.path.join(crt_path, "knn_prep-fin_M09-alive.csv"), encoding='utf-8-sig')
dfc.to_csv(os.path.join(crt_path, "knn_prep-fin_M09.csv"), index=True, encoding='utf-8-sig')

# # With birthdt+7d deathdt data
# dfc_f.to_csv(os.path.join(crt_path, "knn_prep-fin_M04o.csv"), encoding='utf-8-sig')

# # With birthdt+7d deathdt data
# dfc_f.to_csv(os.path.join(crt_path, "knn_prep-fin_M04x.csv"), encoding='utf-8-sig')

In [25]:
dfc

,sex_sys_val,birthdt,bird,admd,apgs1,apgs5,bwei,mage,gran,parn,...,ntetoy_3,ntetoy_1,iperr_1,iperr_2,iperr_3,iperr_4,iperroy_1,iperroy_4,iperroy_3,iperroy_2
0,1,2013-12-18,2013-12-18,2013-12-18,4.0,5.0,1350,29,1,0,...,0,0,1,0,0,0,1,0,0,0
1,1,2013-11-15,2013-11-15,2013-11-15,4.0,5.0,790,35,1,0,...,0,0,1,0,0,0,1,0,0,0
2,1,2013-11-15,2013-11-15,2013-11-15,5.0,6.0,1400,26,1,0,...,0,0,1,0,0,0,1,0,0,0
3,1,2013-11-15,2013-11-15,2013-11-15,6.0,7.0,1340,26,1,0,...,0,0,1,0,0,0,1,0,0,0
4,1,2013-11-11,2013-11-11,2013-11-11,4.0,5.0,1240,36,4,1,...,0,0,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20408,1,2022-11-28,2022-11-28,2022-11-28,5.0,8.0,1350,23,1,0,...,0,0,1,0,0,0,1,0,0,0
20409,1,2022-12-27,2022-12-27,2022-12-27,4.0,7.0,860,38,3,1,...,0,0,1,0,0,0,1,0,0,0
20410,0,2022-12-01,2022-12-01,2022-12-01,5.0,7.0,1910,30,2,0,...,0,0,1,0,0,0,1,0,0,0
20411,1,2022-11-30,2022-11-30,2022-11-30,3.0,7.0,1660,34,1,0,...,0,0,1,0,0,0,1,0,0,0


In [26]:
dfc_alive

,sex_sys_val,birthdt,bird,admd,apgs1,apgs5,bwei,mage,gran,parn,...,ntetoy_3,ntetoy_1,iperr_1,iperr_2,iperr_3,iperr_4,iperroy_1,iperroy_4,iperroy_3,iperroy_2
0,1,2013-12-18,2013-12-18,2013-12-18,4.0,5.0,1350,29,1,0,...,0,0,1,0,0,0,1,0,0,0
1,1,2013-11-15,2013-11-15,2013-11-15,4.0,5.0,790,35,1,0,...,0,0,1,0,0,0,1,0,0,0
2,1,2013-11-15,2013-11-15,2013-11-15,5.0,6.0,1400,26,1,0,...,0,0,1,0,0,0,1,0,0,0
3,1,2013-11-15,2013-11-15,2013-11-15,6.0,7.0,1340,26,1,0,...,0,0,1,0,0,0,1,0,0,0
4,1,2013-11-11,2013-11-11,2013-11-11,4.0,5.0,1240,36,4,1,...,0,0,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20408,1,2022-11-28,2022-11-28,2022-11-28,5.0,8.0,1350,23,1,0,...,0,0,1,0,0,0,1,0,0,0
20409,1,2022-12-27,2022-12-27,2022-12-27,4.0,7.0,860,38,3,1,...,0,0,1,0,0,0,1,0,0,0
20410,0,2022-12-01,2022-12-01,2022-12-01,5.0,7.0,1910,30,2,0,...,0,0,1,0,0,0,1,0,0,0
20411,1,2022-11-30,2022-11-30,2022-11-30,3.0,7.0,1660,34,1,0,...,0,0,1,0,0,0,1,0,0,0


In [25]:
for column in dfc.columns:
    print(f"칼럼 이름: {column}")
    print(f"데이터 타입: {dfc[column].dtype}")
    print(f"유니크한 값:")
    print(dfc[column].unique())
    print("-" * 40)

칼럼 이름: sex_sys_val
데이터 타입: int64
유니크한 값:
[1 0]
----------------------------------------
칼럼 이름: birthdt
데이터 타입: object
유니크한 값:
[datetime.date(2013, 12, 18) datetime.date(2013, 11, 15)
 datetime.date(2013, 11, 11) ... datetime.date(2022, 5, 5)
 datetime.date(2022, 7, 9) datetime.date(2022, 6, 19)]
----------------------------------------
칼럼 이름: bird
데이터 타입: object
유니크한 값:
['2013-12-18' '2013-11-15' '2013-11-11' ... '2022-05-05' '2022-07-09'
 '2022-06-19']
----------------------------------------
칼럼 이름: admd
데이터 타입: object
유니크한 값:
['2013-12-18' '2013-11-15' '2013-11-11' ... '2022-05-05' '2022-07-09'
 '2022-06-19']
----------------------------------------
칼럼 이름: apgs1
데이터 타입: float64
유니크한 값:
[ 4.          5.          6.          3.          7.          1.
  9.          8.          2.          4.75271096  0.         10.        ]
----------------------------------------
칼럼 이름: apgs5
데이터 타입: float64
유니크한 값:
[ 5.          6.          7.          8.          4.          9.
  1.          3.     

In [26]:
drop_cols = []
datetime_cols = ['ntetdt', 'ntetdty', 'date362', 'date36', 'menidt', 'pdaddth', 'pdaddt'
                 , 'birthdt', 'admd', 'birtm', 'bird', 'efydt', 'death_dt', 'iperrdt'
                ]
drop_cols += datetime_cols

rsndth_cols = ['death_CR', 'death_NL', 'death_Inf', 'death_GI', 'death_CA', 'death_Otr']
drop_cols += rsndth_cols
# dfc_f = dfc_f.drop(columns=list(drop_cols))

tgt_cols = [col for col in dfc.columns if col in ['ntet', 'ntety', 'ntetoy'
                                                    , 'SEPS_bef_NTET', 'SEPS_aft_NTET'
                                                    , 'NTET_day', 'PDA_7d_NTET'
                                                    , 'invfpod'
                                                    , 'iperr', 'iperroy'
                                                    , 'SEPS_bef_IPERR', 'SEPS_aft_IPERR'
                                                    , 'IPERR_day', 'PDA_7d_IPERR'
                                                    , 'PDA_7d_IPERRorNTET'
                                                    , 'SEPS_bef_IPERRorNTET', 'SEPS_aft_IPERRorNTET'
                                                   ]
            or any(col.startswith(prefix) for prefix in ['ntet', 'ntety', 'ntetoy'
                                                         , 'SEPS_bef_NTET', 'SEPS_aft_NTET'
                                                         , 'NTET_day', 'PDA_7d_NTET'
                                                         , 'invfpod'
                                                         , 'iperr', 'iperroy'
                                                         , 'SEPS_bef_IPERR', 'SEPS_aft_IPERR'
                                                         , 'IPERR_day', 'PDA_7d_IPERR'
                                                         , 'PDA_7d_IPERRorNTET'
                                                         , 'SEPS_bef_IPERRorNTET', 'SEPS_aft_IPERRorNTET'
                                                        ])
           ]
data = dfc.drop(columns=['death_label'] + list(drop_cols)
#                + list(tgt_cols)
#                + list(drop_cols)
#                + list(datetime_cols)
              )

data

,sex_sys_val,apgs1,apgs5,bwei,mage,gran,parn,bhei,bheiuk,bhead,...,ntetoy_3,ntetoy_1,iperr_1,iperr_2,iperr_3,iperr_4,iperroy_1,iperroy_4,iperroy_3,iperroy_2
0,1,4.0,5.0,1350,29,1,0,39.0,0.0,28.0,...,0,0,1,0,0,0,1,0,0,0
1,1,4.0,5.0,790,35,1,0,34.3,0.0,23.0,...,0,0,1,0,0,0,1,0,0,0
2,1,5.0,6.0,1400,26,1,0,39.0,0.0,28.5,...,0,0,1,0,0,0,1,0,0,0
3,1,6.0,7.0,1340,26,1,0,36.0,0.0,28.0,...,0,0,1,0,0,0,1,0,0,0
4,1,4.0,5.0,1240,36,4,1,42.0,0.0,26.5,...,0,0,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20408,1,5.0,8.0,1350,23,1,0,37.5,0.0,27.5,...,0,0,1,0,0,0,1,0,0,0
20409,1,4.0,7.0,860,38,3,1,33.5,0.0,23.5,...,0,0,1,0,0,0,1,0,0,0
20410,0,5.0,7.0,1910,30,2,0,42.5,0.0,29.0,...,0,0,1,0,0,0,1,0,0,0
20411,1,3.0,7.0,1660,34,1,0,41.5,0.0,28.5,...,0,0,1,0,0,0,1,0,0,0


In [29]:
for column in data.columns:
    print(f"칼럼 이름: {column}")
    print(f"데이터 타입: {data[column].dtype}")
    print(f"유니크한 값:")
    print(data[column].unique())
    print("-" * 40)

칼럼 이름: sex_sys_val
데이터 타입: int64
유니크한 값:
[1 0]
----------------------------------------
칼럼 이름: apgs1
데이터 타입: float64
유니크한 값:
[ 4.          5.          6.          3.          7.          1.
  9.          8.          2.          4.75271096  0.         10.        ]
----------------------------------------
칼럼 이름: apgs5
데이터 타입: float64
유니크한 값:
[ 5.          6.          7.          8.          4.          9.
  1.          3.         10.          2.          6.92736054  0.        ]
----------------------------------------
칼럼 이름: bwei
데이터 타입: int64
유니크한 값:
[1350  790 1400 1340 1240  810 1300 1380  990 1180 1475 1450 1485  960
  910  730 1370 1205 1440 1390 1480 1290 1210 1455 1200 1360 1470 1015
  860  865 1130 1090  920 1260 1220 1320  980  660  600 1270  940 1080
 1190 1490 1040  930  670 1160  480 1330 1250 1460 1310  473  840  850
 1140  610  950 1070 1060  720  680  770 1170  710 1061 1002 1120 1020
  922 1037 1031  765  438 1243  880 1306 1472  708 1376  924  970 1150
 1245 1065 1100 14

In [27]:
analyse_dataframe(data)

,NaN Counts,NaN Ratios (%),Unique Values Count
sex_sys_val,0,0.0,2
apgs1,0,0.0,12
apgs5,0,0.0,12
bwei,0,0.0,902
mage,0,0.0,40
...,...,...,...
iperr_4,0,0.0,2
iperroy_1,0,0.0,2
iperroy_4,0,0.0,2
iperroy_3,0,0.0,2


In [ ]:
## np.nan or '0' Findings

In [31]:
dfc = pd.read_csv(os.path.join(crt_path, "knn_prep-fin_M06.csv"))
dfc.drop(columns = 'Unnamed: 0', inplace = True)
dfc

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_22928\1968373916.py:1: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  dfc = pd.read_csv(os.path.join(crt_path, "knn_prep-fin_M06.csv"))


,sex_sys_val,birthdt,bird,admd,apgs1,apgs5,bwei,mage,gran,parn,...,ntetoy_3,ntetoy_1,iperr_1,iperr_2,iperr_3,iperr_4,iperroy_1,iperroy_4,iperroy_3,iperroy_2
0,1,2013-12-18,2013-12-18,2013-12-18,4.0,5.0,1350,29,1,0,...,0,0,1,0,0,0,1,0,0,0
1,1,2013-11-15,2013-11-15,2013-11-15,4.0,5.0,790,35,1,0,...,0,0,1,0,0,0,1,0,0,0
2,1,2013-11-15,2013-11-15,2013-11-15,5.0,6.0,1400,26,1,0,...,0,0,1,0,0,0,1,0,0,0
3,1,2013-11-15,2013-11-15,2013-11-15,6.0,7.0,1340,26,1,0,...,0,0,1,0,0,0,1,0,0,0
4,1,2013-11-11,2013-11-11,2013-11-11,4.0,5.0,1240,36,4,1,...,0,0,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20347,1,2022-11-28,2022-11-28,2022-11-28,5.0,8.0,1350,23,1,0,...,0,0,1,0,0,0,1,0,0,0
20348,1,2022-12-27,2022-12-27,2022-12-27,4.0,7.0,860,38,3,1,...,0,0,1,0,0,0,1,0,0,0
20349,0,2022-12-01,2022-12-01,2022-12-01,5.0,7.0,1910,30,2,0,...,0,0,1,0,0,0,1,0,0,0
20350,1,2022-11-30,2022-11-30,2022-11-30,3.0,7.0,1660,34,1,0,...,0,0,1,0,0,0,1,0,0,0


In [32]:
for column in dfc.columns:
    print(f"칼럼 이름: {column}")
    print(f"데이터 타입: {dfc[column].dtype}")
    print(f"유니크한 값:")
    print(dfc[column].unique())
    print("-" * 40)

칼럼 이름: sex_sys_val
데이터 타입: int64
유니크한 값:
[1 0]
----------------------------------------
칼럼 이름: birthdt
데이터 타입: object
유니크한 값:
['2013-12-18' '2013-11-15' '2013-11-11' ... '2022-05-05' '2022-07-09'
 '2022-06-19']
----------------------------------------
칼럼 이름: bird
데이터 타입: object
유니크한 값:
['2013-12-18' '2013-11-15' '2013-11-11' ... '2022-05-05' '2022-07-09'
 '2022-06-19']
----------------------------------------
칼럼 이름: admd
데이터 타입: object
유니크한 값:
['2013-12-18' '2013-11-15' '2013-11-11' ... '2022-05-05' '2022-07-09'
 '2022-06-19']
----------------------------------------
칼럼 이름: apgs1
데이터 타입: float64
유니크한 값:
[ 4.          5.          6.          3.          7.          1.
  9.          8.          2.          4.75271096  0.         10.        ]
----------------------------------------
칼럼 이름: apgs5
데이터 타입: float64
유니크한 값:
[ 5.          6.          7.          8.          4.          9.
  1.          3.         10.          2.          6.92736054  0.        ]
---------------------------------

In [33]:
summary = []

for col in dfc.columns:
    dtype = dfc[col].dtype
    total = len(dfc)
    
    # NaN 관련
    nan_count = dfc[col].isna().sum()
    nan_ratio = nan_count / total * 100
    
    # zero 관련
    zero_count = (dfc[col] == 0).sum() if np.issubdtype(dtype, np.number) else np.nan
    zero_ratio = (zero_count / total * 100) if np.issubdtype(dtype, np.number) else np.nan
    
    # binary 관련 (0,1로만 구성되어 있는지 여부)
    # 숫자형이면 고유값을 살펴보고, {0,1}의 부분집합인지 확인
    if np.issubdtype(dtype, np.number):
        unique_vals = dfc[col].dropna().unique()
        is_binary = set(unique_vals).issubset({0, 1})
    else:
        is_binary = False  # 숫자형이 아니면 binary로 간주하지 않음
    
    summary.append({
        'column': col,
        'dtype': dtype,
        'nan_count': nan_count,
        'nan_ratio(%)': round(nan_ratio, 2),
        'zero_count': zero_count,
        'zero_ratio(%)': round(zero_ratio, 2) if not pd.isna(zero_ratio) else np.nan,
        'binary_col': 'Binary' if is_binary else np.nan
    })

df_summary = pd.DataFrame(summary)

df_summary = pd.DataFrame(summary)
df_summary.to_csv(os.path.join(crt_path, "cols_summary_1.csv"), index=True)
df_summary

,column,dtype,nan_count,nan_ratio(%),zero_count,zero_ratio(%),binary_col
0,sex_sys_val,int64,0,0.0,10106.0,49.66,Binary
1,birthdt,object,0,0.0,NaN,NaN,NaN
2,bird,object,0,0.0,NaN,NaN,NaN
3,admd,object,0,0.0,NaN,NaN,NaN
4,apgs1,float64,0,0.0,214.0,1.05,NaN
...,...,...,...,...,...,...,...
151,iperr_4,int64,0,0.0,20345.0,99.97,Binary
152,iperroy_1,int64,0,0.0,455.0,2.24,Binary
153,iperroy_4,int64,0,0.0,20296.0,99.72,Binary
154,iperroy_3,int64,0,0.0,19986.0,98.20,Binary
